In [ ]:
# CELL 1: Package Loading
# ============================================================================

using DifferentialEquations
using Plots
using DataFrames
using CSV
using Statistics
using Printf
using LinearAlgebra
using BlackBoxOptim
using Markdown
using Dates
using StatsPlots
gr()

println("Packages loaded successfully!")

In [ ]:
# CELL 2: Mathematical Model Equations
# ============================================================================

println("\nGENE REGULATORY NETWORK EQUATIONS - COMPLETE")
println("=" ^ 60)

display(Markdown.parse("""
## 11-Gene Network Differential Equations - COMPLETE

### Structure: (Basal + Activations) × (Repressions) - Degradation

### 1. ETS2 (7 activators, 3 repressors)
```math
\\frac{dETS2}{dt} = \\left(\\alpha_1 + \\gamma_1 \\frac{HIF1A^{n_1}}{K_1^{n_1} + HIF1A^{n_1}} + \\gamma_2 \\frac{ETS1^{n_2}}{K_2^{n_2} + ETS1^{n_2}} + \\gamma_3 \\frac{JUN^{n_3}}{K_3^{n_3} + JUN^{n_3}} + \\gamma_4 \\frac{MYC^{n_4}}{K_4^{n_4} + MYC^{n_4}} + \\gamma_5 \\frac{STAT1^{n_5}}{K_5^{n_5} + STAT1^{n_5}} + \\gamma_6 \\frac{STAT3^{n_6}}{K_6^{n_6} + STAT3^{n_6}} + \\gamma_7 \\frac{TP53^{n_7}}{K_7^{n_7} + TP53^{n_7}}\\right) \\times \\left(\\frac{\\gamma_{r1}}{1 + \\left(\\frac{HIF1A}{K_{r1}}\\right)^{n_{r1}}} \\cdot \\frac{\\gamma_{r2}}{1 + \\left(\\frac{STAT3}{K_{r2}}\\right)^{n_{r2}}} \\cdot \\frac{\\gamma_{r3}}{1 + \\left(\\frac{VitD}{K_{r3}}\\right)^{n_{r3}}}\\right) - \\delta_1 \\cdot ETS2
```

### 2. FOS (10 activators, 7 repressors)
```math
\\frac{dFOS}{dt} = \\left(\\alpha_2 + \\gamma_8 \\frac{ETS2^{n_8}}{K_8^{n_8} + ETS2^{n_8}} + \\gamma_9 \\frac{HIF1A^{n_9}}{K_9^{n_9} + HIF1A^{n_9}} + \\gamma_{10} \\frac{EGR1^{n_{10}}}{K_{10}^{n_{10}} + EGR1^{n_{10}}} + \\gamma_{11} \\frac{ETS1^{n_{11}}}{K_{11}^{n_{11}} + ETS1^{n_{11}}} + \\gamma_{12} \\frac{JUN^{n_{12}}}{K_{12}^{n_{12}} + JUN^{n_{12}}} + \\gamma_{13} \\frac{MYC^{n_{13}}}{K_{13}^{n_{13}} + MYC^{n_{13}}} + \\gamma_{14} \\frac{STAT1^{n_{14}}}{K_{14}^{n_{14}} + STAT1^{n_{14}}} + \\gamma_{15} \\frac{STAT3^{n_{15}}}{K_{15}^{n_{15}} + STAT3^{n_{15}}} + \\gamma_{16} \\frac{TP53^{n_{16}}}{K_{16}^{n_{16}} + TP53^{n_{16}}} + \\gamma_{17} \\frac{VitD^{n_{17}}}{K_{17}^{n_{17}} + VitD^{n_{17}}}\\right) \\times \\left(\\frac{\\gamma_{r4}}{1 + \\left(\\frac{HIF1A}{K_{r4}}\\right)^{n_{r4}}} \\cdot \\frac{\\gamma_{r5}}{1 + \\left(\\frac{JUN}{K_{r5}}\\right)^{n_{r5}}} \\cdot \\frac{\\gamma_{r6}}{1 + \\left(\\frac{MYC}{K_{r6}}\\right)^{n_{r6}}} \\cdot \\frac{\\gamma_{r7}}{1 + \\left(\\frac{STAT1}{K_{r7}}\\right)^{n_{r7}}} \\cdot \\frac{\\gamma_{r8}}{1 + \\left(\\frac{STAT3}{K_{r8}}\\right)^{n_{r8}}} \\cdot \\frac{\\gamma_{r9}}{1 + \\left(\\frac{TP53}{K_{r9}}\\right)^{n_{r9}}} \\cdot \\frac{\\gamma_{r10}}{1 + \\left(\\frac{VitD}{K_{r10}}\\right)^{n_{r10}}}\\right) - \\delta_2 \\cdot FOS
```

### 3. HIF1A (5 activators, 1 repressor)
```math
\\frac{dHIF1A}{dt} = \\left(\\alpha_3 + \\gamma_{18} \\frac{EGR1^{n_{18}}}{K_{18}^{n_{18}} + EGR1^{n_{18}}} + \\gamma_{19} \\frac{MYC^{n_{19}}}{K_{19}^{n_{19}} + MYC^{n_{19}}} + \\gamma_{20} \\frac{NFKB1^{n_{20}}}{K_{20}^{n_{20}} + NFKB1^{n_{20}}} + \\gamma_{21} \\frac{STAT3^{n_{21}}}{K_{21}^{n_{21}} + STAT3^{n_{21}}} + \\gamma_{22} \\frac{VitD^{n_{22}}}{K_{22}^{n_{22}} + VitD^{n_{22}}}\\right) \\times \\left(\\frac{\\gamma_{r11}}{1 + \\left(\\frac{VitD}{K_{r11}}\\right)^{n_{r11}}}\\right) - \\delta_3 \\cdot HIF1A
```

### 4. EGR1 (8 activators, 3 repressors)
```math
\\frac{dEGR1}{dt} = \\left(\\alpha_4 + \\gamma_{23} \\frac{ETS2^{n_{23}}}{K_{23}^{n_{23}} + ETS2^{n_{23}}} + \\gamma_{24} \\frac{HIF1A^{n_{24}}}{K_{24}^{n_{24}} + HIF1A^{n_{24}}} + \\gamma_{25} \\frac{ETS1^{n_{25}}}{K_{25}^{n_{25}} + ETS1^{n_{25}}} + \\gamma_{26} \\frac{MYC^{n_{26}}}{K_{26}^{n_{26}} + MYC^{n_{26}}} + \\gamma_{27} \\frac{STAT1^{n_{27}}}{K_{27}^{n_{27}} + STAT1^{n_{27}}} + \\gamma_{28} \\frac{STAT3^{n_{28}}}{K_{28}^{n_{28}} + STAT3^{n_{28}}} + \\gamma_{29} \\frac{TP53^{n_{29}}}{K_{29}^{n_{29}} + TP53^{n_{29}}} + \\gamma_{30} \\frac{VitD^{n_{30}}}{K_{30}^{n_{30}} + VitD^{n_{30}}}\\right) \\times \\left(\\frac{\\gamma_{r12}}{1 + \\left(\\frac{HIF1A}{K_{r12}}\\right)^{n_{r12}}} \\cdot \\frac{\\gamma_{r13}}{1 + \\left(\\frac{STAT3}{K_{r13}}\\right)^{n_{r13}}} \\cdot \\frac{\\gamma_{r14}}{1 + \\left(\\frac{VitD}{K_{r14}}\\right)^{n_{r14}}}\\right) - \\delta_4 \\cdot EGR1
```

### 5. ETS1 (4 activators, 8 repressors)
```math
\\frac{dETS1}{dt} = \\left(\\alpha_5 + \\gamma_{31} \\frac{HIF1A^{n_{31}}}{K_{31}^{n_{31}} + HIF1A^{n_{31}}} + \\gamma_{32} \\frac{JUN^{n_{32}}}{K_{32}^{n_{32}} + JUN^{n_{32}}} + \\gamma_{33} \\frac{STAT3^{n_{33}}}{K_{33}^{n_{33}} + STAT3^{n_{33}}} + \\gamma_{34} \\frac{VitD^{n_{34}}}{K_{34}^{n_{34}} + VitD^{n_{34}}}\\right) \\times \\left(\\frac{\\gamma_{r15}}{1 + \\left(\\frac{FOS}{K_{r15}}\\right)^{n_{r15}}} \\cdot \\frac{\\gamma_{r16}}{1 + \\left(\\frac{HIF1A}{K_{r16}}\\right)^{n_{r16}}} \\cdot \\frac{\\gamma_{r17}}{1 + \\left(\\frac{JUN}{K_{r17}}\\right)^{n_{r17}}} \\cdot \\frac{\\gamma_{r18}}{1 + \\left(\\frac{MYC}{K_{r18}}\\right)^{n_{r18}}} \\cdot \\frac{\\gamma_{r19}}{1 + \\left(\\frac{STAT1}{K_{r19}}\\right)^{n_{r19}}} \\cdot \\frac{\\gamma_{r20}}{1 + \\left(\\frac{STAT3}{K_{r20}}\\right)^{n_{r20}}} \\cdot \\frac{\\gamma_{r21}}{1 + \\left(\\frac{TP53}{K_{r21}}\\right)^{n_{r21}}} \\cdot \\frac{\\gamma_{r22}}{1 + \\left(\\frac{VitD}{K_{r22}}\\right)^{n_{r22}}}\\right) - \\delta_5 \\cdot ETS1
```

### 6. JUN (7 activators, 7 repressors)
```math
\\frac{dJUN}{dt} = \\left(\\alpha_6 + \\gamma_{35} \\frac{HIF1A^{n_{35}}}{K_{35}^{n_{35}} + HIF1A^{n_{35}}} + \\gamma_{36} \\frac{EGR1^{n_{36}}}{K_{36}^{n_{36}} + EGR1^{n_{36}}} + \\gamma_{37} \\frac{MYC^{n_{37}}}{K_{37}^{n_{37}} + MYC^{n_{37}}} + \\gamma_{38} \\frac{STAT1^{n_{38}}}{K_{38}^{n_{38}} + STAT1^{n_{38}}} + \\gamma_{39} \\frac{STAT3^{n_{39}}}{K_{39}^{n_{39}} + STAT3^{n_{39}}} + \\gamma_{40} \\frac{TP53^{n_{40}}}{K_{40}^{n_{40}} + TP53^{n_{40}}} + \\gamma_{41} \\frac{VitD^{n_{41}}}{K_{41}^{n_{41}} + VitD^{n_{41}}}\\right) \\times \\left(\\frac{\\gamma_{r23}}{1 + \\left(\\frac{FOS}{K_{r23}}\\right)^{n_{r23}}} \\cdot \\frac{\\gamma_{r24}}{1 + \\left(\\frac{HIF1A}{K_{r24}}\\right)^{n_{r24}}} \\cdot \\frac{\\gamma_{r25}}{1 + \\left(\\frac{MYC}{K_{r25}}\\right)^{n_{r25}}} \\cdot \\frac{\\gamma_{r26}}{1 + \\left(\\frac{STAT1}{K_{r26}}\\right)^{n_{r26}}} \\cdot \\frac{\\gamma_{r27}}{1 + \\left(\\frac{STAT3}{K_{r27}}\\right)^{n_{r27}}} \\cdot \\frac{\\gamma_{r28}}{1 + \\left(\\frac{TP53}{K_{r28}}\\right)^{n_{r28}}} \\cdot \\frac{\\gamma_{r29}}{1 + \\left(\\frac{VitD}{K_{r29}}\\right)^{n_{r29}}}\\right) - \\delta_6 \\cdot JUN
```

### 7. MYC (8 activators, 6 repressors)
```math
\\frac{dMYC}{dt} = \\left(\\alpha_7 + \\gamma_{42} \\frac{ETS2^{n_{42}}}{K_{42}^{n_{42}} + ETS2^{n_{42}}} + \\gamma_{43} \\frac{HIF1A^{n_{43}}}{K_{43}^{n_{43}} + HIF1A^{n_{43}}} + \\gamma_{44} \\frac{EGR1^{n_{44}}}{K_{44}^{n_{44}} + EGR1^{n_{44}}} + \\gamma_{45} \\frac{NFKB1^{n_{45}}}{K_{45}^{n_{45}} + NFKB1^{n_{45}}} + \\gamma_{46} \\frac{STAT1^{n_{46}}}{K_{46}^{n_{46}} + STAT1^{n_{46}}} + \\gamma_{47} \\frac{STAT3^{n_{47}}}{K_{47}^{n_{47}} + STAT3^{n_{47}}} + \\gamma_{48} \\frac{TP53^{n_{48}}}{K_{48}^{n_{48}} + TP53^{n_{48}}} + \\gamma_{49} \\frac{VitD^{n_{49}}}{K_{49}^{n_{49}} + VitD^{n_{49}}}\\right) \\times \\left(\\frac{\\gamma_{r30}}{1 + \\left(\\frac{FOS}{K_{r30}}\\right)^{n_{r30}}} \\cdot \\frac{\\gamma_{r31}}{1 + \\left(\\frac{HIF1A}{K_{r31}}\\right)^{n_{r31}}} \\cdot \\frac{\\gamma_{r32}}{1 + \\left(\\frac{JUN}{K_{r32}}\\right)^{n_{r32}}} \\cdot \\frac{\\gamma_{r33}}{1 + \\left(\\frac{STAT3}{K_{r33}}\\right)^{n_{r33}}} \\cdot \\frac{\\gamma_{r34}}{1 + \\left(\\frac{TP53}{K_{r34}}\\right)^{n_{r34}}} \\cdot \\frac{\\gamma_{r35}}{1 + \\left(\\frac{VitD}{K_{r35}}\\right)^{n_{r35}}}\\right) - \\delta_7 \\cdot MYC
```

### 8. NFKB1 (7 activators, 4 repressors)
```math
\\frac{dNFKB1}{dt} = \\left(\\alpha_8 + \\gamma_{50} \\frac{HIF1A^{n_{50}}}{K_{50}^{n_{50}} + HIF1A^{n_{50}}} + \\gamma_{51} \\frac{EGR1^{n_{51}}}{K_{51}^{n_{51}} + EGR1^{n_{51}}} + \\gamma_{52} \\frac{ETS1^{n_{52}}}{K_{52}^{n_{52}} + ETS1^{n_{52}}} + \\gamma_{53} \\frac{MYC^{n_{53}}}{K_{53}^{n_{53}} + MYC^{n_{53}}} + \\gamma_{54} \\frac{STAT1^{n_{54}}}{K_{54}^{n_{54}} + STAT1^{n_{54}}} + \\gamma_{55} \\frac{STAT3^{n_{55}}}{K_{55}^{n_{55}} + STAT3^{n_{55}}} + \\gamma_{56} \\frac{TP53^{n_{56}}}{K_{56}^{n_{56}} + TP53^{n_{56}}}\\right) \\times \\left(\\frac{\\gamma_{r36}}{1 + \\left(\\frac{FOS}{K_{r36}}\\right)^{n_{r36}}} \\cdot \\frac{\\gamma_{r37}}{1 + \\left(\\frac{HIF1A}{K_{r37}}\\right)^{n_{r37}}} \\cdot \\frac{\\gamma_{r38}}{1 + \\left(\\frac{STAT3}{K_{r38}}\\right)^{n_{r38}}} \\cdot \\frac{\\gamma_{r39}}{1 + \\left(\\frac{VitD}{K_{r39}}\\right)^{n_{r39}}}\\right) - \\delta_8 \\cdot NFKB1
```

### 9. STAT1 (3 activators, 0 repressors)
```math
\\frac{dSTAT1}{dt} = \\left(\\alpha_9 + \\gamma_{57} \\frac{STAT3^{n_{57}}}{K_{57}^{n_{57}} + STAT3^{n_{57}}} + \\gamma_{58} \\frac{TP53^{n_{58}}}{K_{58}^{n_{58}} + TP53^{n_{58}}} + \\gamma_{59} \\frac{VitD^{n_{59}}}{K_{59}^{n_{59}} + VitD^{n_{59}}}\\right) - \\delta_9 \\cdot STAT1
```

### 10. STAT3 (2 activators, 1 repressor)
```math
\\frac{dSTAT3}{dt} = \\left(\\alpha_{10} + \\gamma_{60} \\frac{STAT1^{n_{60}}}{K_{60}^{n_{60}} + STAT1^{n_{60}}} + \\gamma_{61} \\frac{VitD^{n_{61}}}{K_{61}^{n_{61}} + VitD^{n_{61}}}\\right) \\times \\left(\\frac{\\gamma_{r40}}{1 + \\left(\\frac{TP53}{K_{r40}}\\right)^{n_{r40}}}\\right) - \\delta_{10} \\cdot STAT3
```

### 11. TP53 (10 activators, 6 repressors)
```math
\\frac{dTP53}{dt} = \\left(\\alpha_{11} + \\gamma_{62} \\frac{ETS2^{n_{62}}}{K_{62}^{n_{62}} + ETS2^{n_{62}}} + \\gamma_{63} \\frac{FOS^{n_{63}}}{K_{63}^{n_{63}} + FOS^{n_{63}}} + \\gamma_{64} \\frac{HIF1A^{n_{64}}}{K_{64}^{n_{64}} + HIF1A^{n_{64}}} + \\gamma_{65} \\frac{EGR1^{n_{65}}}{K_{65}^{n_{65}} + EGR1^{n_{65}}} + \\gamma_{66} \\frac{ETS1^{n_{66}}}{K_{66}^{n_{66}} + ETS1^{n_{66}}} + \\gamma_{67} \\frac{MYC^{n_{67}}}{K_{67}^{n_{67}} + MYC^{n_{67}}} + \\gamma_{68} \\frac{NFKB1^{n_{68}}}{K_{68}^{n_{68}} + NFKB1^{n_{68}}} + \\gamma_{69} \\frac{STAT1^{n_{69}}}{K_{69}^{n_{69}} + STAT1^{n_{69}}} + \\gamma_{70} \\frac{STAT3^{n_{70}}}{K_{70}^{n_{70}} + STAT3^{n_{70}}} + \\gamma_{71} \\frac{VitD^{n_{71}}}{K_{71}^{n_{71}} + VitD^{n_{71}}}\\right) \\times \\left(\\frac{\\gamma_{r41}}{1 + \\left(\\frac{HIF1A}{K_{r41}}\\right)^{n_{r41}}} \\cdot \\frac{\\gamma_{r42}}{1 + \\left(\\frac{JUN}{K_{r42}}\\right)^{n_{r42}}} \\cdot \\frac{\\gamma_{r43}}{1 + \\left(\\frac{MYC}{K_{r43}}\\right)^{n_{r43}}} \\cdot \\frac{\\gamma_{r44}}{1 + \\left(\\frac{STAT1}{K_{r44}}\\right)^{n_{r44}}} \\cdot \\frac{\\gamma_{r45}}{1 + \\left(\\frac{STAT3}{K_{r45}}\\right)^{n_{r45}}} \\cdot \\frac{\\gamma_{r46}}{1 + \\left(\\frac{VitD}{K_{r46}}\\right)^{n_{r46}}}\\right) - \\delta_{11} \\cdot TP53
```

### 12. Vitamin D (FIXED PARAMETERS)
```math
\\frac{dVitD}{dt} = k_i - d_i \\cdot VitD
```

---

**Parameters Summary:**
- α₁-α₁₁: Basal production rates (11)
- δ₁-δ₁₁: Degradation rates (11)
- γ₁-γ₇₁: Activation coefficients (71)
- γᵣ₁-γᵣ₄₆: Repression coefficients (46)
- K₁-K₇₁: Activation Hill K values (71)
- Kᵣ₁-Kᵣ₄₆: Repression Hill K values (46)
- n₁-n₇₁: Activation Hill n values (71)
- nᵣ₁-nᵣ₄₆: Repression Hill n values (46)
- kᵢ, dᵢ: VitD synthesis/degradation (FIXED, not optimized)

**Total: 373 optimizable parameters + 2 fixed VitD parameters per replicate**

**Total activations: 71**
**Total repressions: 46**
**Total regulatory interactions: 117**
"""))

# Save equations to file
open("Model_Equations.txt", "w") do f
    write(f, "GENE REGULATORY NETWORK EQUATIONS\n")
    write(f, "="^60 * "\n\n")
    write(f, "Structure: (Basal + Activations) × (Repressions) - Degradation\n\n")
    write(f, "1. dETS2/dt = (α₁ + Σγᵢ·Hill(activators)) × Πγᵣⱼ·HillRep(repressors) - δ₁·ETS2\n")
    write(f, "2. dFOS/dt = (α₂ + Σγᵢ·Hill(activators)) × Πγᵣⱼ·HillRep(repressors) - δ₂·FOS\n")
    write(f, "3. dHIF1A/dt = (α₃ + Σγᵢ·Hill(activators)) × Πγᵣⱼ·HillRep(repressors) - δ₃·HIF1A\n")
    write(f, "4. dEGR1/dt = (α₄ + Σγᵢ·Hill(activators)) × Πγᵣⱼ·HillRep(repressors) - δ₄·EGR1\n")
    write(f, "5. dETS1/dt = (α₅ + Σγᵢ·Hill(activators)) × Πγᵣⱼ·HillRep(repressors) - δ₅·ETS1\n")
    write(f, "6. dJUN/dt = (α₆ + Σγᵢ·Hill(activators)) × Πγᵣⱼ·HillRep(repressors) - δ₆·JUN\n")
    write(f, "7. dMYC/dt = (α₇ + Σγᵢ·Hill(activators)) × Πγᵣⱼ·HillRep(repressors) - δ₇·MYC\n")
    write(f, "8. dNFKB1/dt = (α₈ + Σγᵢ·Hill(activators)) × Πγᵣⱼ·HillRep(repressors) - δ₈·NFKB1\n")
    write(f, "9. dSTAT1/dt = α₉ + Σγᵢ·Hill(activators) - δ₉·STAT1\n")
    write(f, "10. dSTAT3/dt = (α₁₀ + Σγᵢ·Hill(activators)) × Πγᵣⱼ·HillRep(repressors) - δ₁₀·STAT3\n")
    write(f, "11. dTP53/dt = (α₁₁ + Σγᵢ·Hill(activators)) × Πγᵣⱼ·HillRep(repressors) - δ₁₁·TP53\n")
    write(f, "12. dVitD/dt = k - d·VitD  (FIXED parameters)\n\n")
    write(f, "Hill Activation: H⁺(x) = γ·xⁿ/(Kⁿ + xⁿ)\n")
    write(f, "Hill Repression: H⁻(x) = γᵣ/(1 + (x/Kᵣ)ⁿʳ)\n\n")
    write(f, "Total optimizable parameters: 373\n")
    write(f, "Total regulatory interactions: 117 (71 activations + 46 repressions)\n")
end
println(" Equations saved to Model_Equations.txt")

In [ ]:
# CELL 3: Load Experimental Data
# ============================================================================
println("\nLOADING DATA")
println("=" ^ 20)

function convert_european_decimal(value_str)
    if value_str isa AbstractString
        return parse(Float64, replace(String(value_str), "," => "."))
    elseif ismissing(value_str) || value_str == ""
        return NaN
    else
        return Float64(value_str)
    end
end

gene_names = ["ETS2", "FOS", "HIF1A", "EGR1", "ETS1", "JUN", "MYC", "NFKB1", "STAT1", "STAT3", "TP53"]

raw_data = CSV.read("gene_exp_filtered_ind_12gene.csv", DataFrame; delim=';', header=true)
println("Loaded: $(nrow(raw_data)) rows × $(ncol(raw_data)) columns")

experimental_data = Dict{String, Matrix{Float64}}()
vitd_measured = Dict{String, Tuple{Float64, Float64}}()

for rep in ["R1", "R2", "R3"]
    experimental_data[rep] = zeros(11, 2)
    
    for (i, gene) in enumerate(gene_names)
        gene_row = filter(r -> r.hgnc_symbol == gene, raw_data)
        if nrow(gene_row) > 0
            row = gene_row[1, :]
            t0_cols = filter(c -> startswith(string(c), "$(rep)_0_"), names(raw_data))
            t0_vals = [convert_european_decimal(row[c]) for c in t0_cols]
            t0_vals = filter(v -> !isnan(v) && v > 0, t0_vals)
            t24_cols = filter(c -> startswith(string(c), "$(rep)_24_"), names(raw_data))
            t24_vals = [convert_european_decimal(row[c]) for c in t24_cols]
            t24_vals = filter(v -> !isnan(v) && v > 0, t24_vals)
            experimental_data[rep][i, 1] = isempty(t0_vals) ? 1000.0 : mean(t0_vals)
            experimental_data[rep][i, 2] = isempty(t24_vals) ? 1000.0 : mean(t24_vals)
        end
    end
    
    # LOAD MEASURED VitD
    vitd_row = filter(r -> r.hgnc_symbol == "VitD", raw_data)
    if nrow(vitd_row) > 0
        row = vitd_row[1, :]
        t0_cols = filter(c -> startswith(string(c), "$(rep)_0_"), names(raw_data))
        t0_vals = [convert_european_decimal(row[c]) for c in t0_cols]
        t0_vals = filter(v -> !isnan(v) && v > 0, t0_vals)
        t24_cols = filter(c -> startswith(string(c), "$(rep)_24_"), names(raw_data))
        t24_vals = [convert_european_decimal(row[c]) for c in t24_cols]
        t24_vals = filter(v -> !isnan(v) && v > 0, t24_vals)
        vitd_measured[rep] = (isempty(t0_vals) ? 25.0 : mean(t0_vals),
                             isempty(t24_vals) ? 35.0 : mean(t24_vals))
    else
        vitd_measured[rep] = (25.0, 35.0)
    end
end

println("\n Processed genes: $gene_names")
println("\nSample data:")
for rep in ["R1", "R2", "R3"]
    println("$rep:")
    for (i, gene) in enumerate(gene_names)
        t0, t24 = experimental_data[rep][i, 1], experimental_data[rep][i, 2]
        change = ((t24 - t0) / t0) * 100
        @printf("  %s: %.0f → %.0f (%.1f%%)\n", gene, t0, t24, change)
    end
end

println("\n Measured VitD levels:")
for rep in ["R1", "R2", "R3"]
    v0, v24 = vitd_measured[rep]
    @printf("  %s: %.1f → %.1f nmol/L\n", rep, v0, v24)
end

In [ ]:
# CELL 4: Model Implementation
# ============================================================================

println("\nDEFINING COMPLETE MODEL")
println("=" ^ 30)

# VitD parameters (FIXED - not part of optimization)
vitd_params = Dict("R1" => (k=0.3684, d=0.000557), "R2" => (k=0.1968, d=0.000122), "R3" => (k=0.2787, d=0.000130))

function gene_model!(du, u, p, t, rep)
    ETS2, FOS, HIF1A, EGR1, ETS1, JUN, MYC, NFKB1, STAT1, STAT3, TP53, VitD = max.(u, 1e-6)
    
    ki, di = vitd_params[rep].k, vitd_params[rep].d
    
    α = p[1:11]
    δ = p[12:22]
    γ = p[23:93]
    γr = p[94:139]
    K = p[140:210]
    n = p[211:281]
    Kr = p[282:327]
    nr = p[328:373]
    
    function hill_act(x, γ_val, K_val, n_val)
        K_val = max(K_val, 1.0)
        n_val = clamp(n_val, 0.5, 5.0)
        return γ_val * (x^n_val) / (K_val^n_val + x^n_val)
    end
    
    function hill_rep(x, γr_val, Kr_val, nr_val)
        Kr_val = max(Kr_val, 1.0)
        nr_val = clamp(nr_val, 0.5, 5.0)
        return γr_val / (1.0 + (x/Kr_val)^nr_val)
    end
    
    # 1. ETS2
    act = α[1] + hill_act(HIF1A,γ[1],K[1],n[1]) + hill_act(ETS1,γ[2],K[2],n[2]) +
          hill_act(JUN,γ[3],K[3],n[3]) + hill_act(MYC,γ[4],K[4],n[4]) +
          hill_act(STAT1,γ[5],K[5],n[5]) + hill_act(STAT3,γ[6],K[6],n[6]) + hill_act(TP53,γ[7],K[7],n[7])
    rep_term = hill_rep(HIF1A,γr[1],Kr[1],nr[1]) * hill_rep(STAT3,γr[2],Kr[2],nr[2]) * hill_rep(VitD,γr[3],Kr[3],nr[3])
    du[1] = act * rep_term - δ[1] * ETS2
    
    # 2. FOS
    act = α[2] + hill_act(ETS2,γ[8],K[8],n[8]) + hill_act(HIF1A,γ[9],K[9],n[9]) +
          hill_act(EGR1,γ[10],K[10],n[10]) + hill_act(ETS1,γ[11],K[11],n[11]) +
          hill_act(JUN,γ[12],K[12],n[12]) + hill_act(MYC,γ[13],K[13],n[13]) +
          hill_act(STAT1,γ[14],K[14],n[14]) + hill_act(STAT3,γ[15],K[15],n[15]) +
          hill_act(TP53,γ[16],K[16],n[16]) + hill_act(VitD,γ[17],K[17],n[17])
    rep_term = hill_rep(HIF1A,γr[4],Kr[4],nr[4]) * hill_rep(JUN,γr[5],Kr[5],nr[5]) *
               hill_rep(MYC,γr[6],Kr[6],nr[6]) * hill_rep(STAT1,γr[7],Kr[7],nr[7]) *
               hill_rep(STAT3,γr[8],Kr[8],nr[8]) * hill_rep(TP53,γr[9],Kr[9],nr[9]) * hill_rep(VitD,γr[10],Kr[10],nr[10])
    du[2] = act * rep_term - δ[2] * FOS
    
    # 3. HIF1A
    act = α[3] + hill_act(EGR1,γ[18],K[18],n[18]) + hill_act(MYC,γ[19],K[19],n[19]) +
          hill_act(NFKB1,γ[20],K[20],n[20]) + hill_act(STAT3,γ[21],K[21],n[21]) + hill_act(VitD,γ[22],K[22],n[22])
    rep_term = hill_rep(VitD,γr[11],Kr[11],nr[11])
    du[3] = act * rep_term - δ[3] * HIF1A
    
    # 4. EGR1
    act = α[4] + hill_act(ETS2,γ[23],K[23],n[23]) + hill_act(HIF1A,γ[24],K[24],n[24]) +
          hill_act(ETS1,γ[25],K[25],n[25]) + hill_act(MYC,γ[26],K[26],n[26]) +
          hill_act(STAT1,γ[27],K[27],n[27]) + hill_act(STAT3,γ[28],K[28],n[28]) +
          hill_act(TP53,γ[29],K[29],n[29]) + hill_act(VitD,γ[30],K[30],n[30])
    rep_term = hill_rep(HIF1A,γr[12],Kr[12],nr[12]) * hill_rep(STAT3,γr[13],Kr[13],nr[13]) * hill_rep(VitD,γr[14],Kr[14],nr[14])
    du[4] = act * rep_term - δ[4] * EGR1
    
    # 5. ETS1
    act = α[5] + hill_act(HIF1A,γ[31],K[31],n[31]) + hill_act(JUN,γ[32],K[32],n[32]) +
          hill_act(STAT3,γ[33],K[33],n[33]) + hill_act(VitD,γ[34],K[34],n[34])
    rep_term = hill_rep(FOS,γr[15],Kr[15],nr[15]) * hill_rep(HIF1A,γr[16],Kr[16],nr[16]) *
               hill_rep(JUN,γr[17],Kr[17],nr[17]) * hill_rep(MYC,γr[18],Kr[18],nr[18]) *
               hill_rep(STAT1,γr[19],Kr[19],nr[19]) * hill_rep(STAT3,γr[20],Kr[20],nr[20]) *
               hill_rep(TP53,γr[21],Kr[21],nr[21]) * hill_rep(VitD,γr[22],Kr[22],nr[22])
    du[5] = act * rep_term - δ[5] * ETS1
    
    # 6. JUN
    act = α[6] + hill_act(HIF1A,γ[35],K[35],n[35]) + hill_act(EGR1,γ[36],K[36],n[36]) +
          hill_act(MYC,γ[37],K[37],n[37]) + hill_act(STAT1,γ[38],K[38],n[38]) +
          hill_act(STAT3,γ[39],K[39],n[39]) + hill_act(TP53,γ[40],K[40],n[40]) + hill_act(VitD,γ[41],K[41],n[41])
    rep_term = hill_rep(FOS,γr[23],Kr[23],nr[23]) * hill_rep(HIF1A,γr[24],Kr[24],nr[24]) *
               hill_rep(MYC,γr[25],Kr[25],nr[25]) * hill_rep(STAT1,γr[26],Kr[26],nr[26]) *
               hill_rep(STAT3,γr[27],Kr[27],nr[27]) * hill_rep(TP53,γr[28],Kr[28],nr[28]) * hill_rep(VitD,γr[29],Kr[29],nr[29])
    du[6] = act * rep_term - δ[6] * JUN
    
    # 7. MYC
    act = α[7] + hill_act(ETS2,γ[42],K[42],n[42]) + hill_act(HIF1A,γ[43],K[43],n[43]) +
          hill_act(EGR1,γ[44],K[44],n[44]) + hill_act(NFKB1,γ[45],K[45],n[45]) +
          hill_act(STAT1,γ[46],K[46],n[46]) + hill_act(STAT3,γ[47],K[47],n[47]) +
          hill_act(TP53,γ[48],K[48],n[48]) + hill_act(VitD,γ[49],K[49],n[49])
    rep_term = hill_rep(FOS,γr[30],Kr[30],nr[30]) * hill_rep(HIF1A,γr[31],Kr[31],nr[31]) *
               hill_rep(JUN,γr[32],Kr[32],nr[32]) * hill_rep(STAT3,γr[33],Kr[33],nr[33]) *
               hill_rep(TP53,γr[34],Kr[34],nr[34]) * hill_rep(VitD,γr[35],Kr[35],nr[35])
    du[7] = act * rep_term - δ[7] * MYC
    
    # 8. NFKB1
    act = α[8] + hill_act(HIF1A,γ[50],K[50],n[50]) + hill_act(EGR1,γ[51],K[51],n[51]) +
          hill_act(ETS1,γ[52],K[52],n[52]) + hill_act(MYC,γ[53],K[53],n[53]) +
          hill_act(STAT1,γ[54],K[54],n[54]) + hill_act(STAT3,γ[55],K[55],n[55]) + hill_act(TP53,γ[56],K[56],n[56])
    rep_term = hill_rep(FOS,γr[36],Kr[36],nr[36]) * hill_rep(HIF1A,γr[37],Kr[37],nr[37]) *
               hill_rep(STAT3,γr[38],Kr[38],nr[38]) * hill_rep(VitD,γr[39],Kr[39],nr[39])
    du[8] = act * rep_term - δ[8] * NFKB1
    
    # 9. STAT1 (no repressors)
    du[9] = (α[9] + hill_act(STAT3,γ[57],K[57],n[57]) + hill_act(TP53,γ[58],K[58],n[58]) +
             hill_act(VitD,γ[59],K[59],n[59])) - δ[9] * STAT1
    
    # 10. STAT3
    act = α[10] + hill_act(STAT1,γ[60],K[60],n[60]) + hill_act(VitD,γ[61],K[61],n[61])
    rep_term = hill_rep(TP53,γr[40],Kr[40],nr[40])
    du[10] = act * rep_term - δ[10] * STAT3
    
    # 11. TP53
    act = α[11] + hill_act(ETS2,γ[62],K[62],n[62]) + hill_act(FOS,γ[63],K[63],n[63]) +
          hill_act(HIF1A,γ[64],K[64],n[64]) + hill_act(EGR1,γ[65],K[65],n[65]) +
          hill_act(ETS1,γ[66],K[66],n[66]) + hill_act(MYC,γ[67],K[67],n[67]) +
          hill_act(NFKB1,γ[68],K[68],n[68]) + hill_act(STAT1,γ[69],K[69],n[69]) +
          hill_act(STAT3,γ[70],K[70],n[70]) + hill_act(VitD,γ[71],K[71],n[71])
    rep_term = hill_rep(HIF1A,γr[41],Kr[41],nr[41]) * hill_rep(JUN,γr[42],Kr[42],nr[42]) *
               hill_rep(MYC,γr[43],Kr[43],nr[43]) * hill_rep(STAT1,γr[44],Kr[44],nr[44]) *
               hill_rep(STAT3,γr[45],Kr[45],nr[45]) * hill_rep(VitD,γr[46],Kr[46],nr[46])
    du[11] = act * rep_term - δ[11] * TP53
    
    # 12. VitD (FIXED parameters)
    du[12] = ki - di * VitD
    
    for i in 1:12
        du[i] = clamp(du[i], -1e6, 1e6)
    end
    nothing
end

println("Model defined: 12 equations, 373 optimizable parameters")

In [ ]:
# CELL 5: Create Parameters and Initial Conditions
# ============================================================================
println("\nSETTING PARAMETERS")
println("=" ^ 25)

function create_parameters_improved(replicate)
    return [
        fill(1.0, 11)...,       # α: Basal rates
        fill(0.0001, 11)...,    # δ: Degradation rates
        fill(10.0, 71)...,      # γ: Activation coefficients
        fill(5.0, 46)...,       # γr: Repression coefficients
        fill(10.0, 71)...,      # K: Hill K values
        fill(2.0, 71)...,       # n: Hill n values
        fill(10.0, 46)...,      # Kr: Hill K values for repression
        fill(2.0, 46)...        # nr: Hill n values for repression
    ]
end

params = Dict{String, Vector{Float64}}()
u0 = Dict{String, Vector{Float64}}()
optimized_params = Dict{String, Any}()

for rep in ["R1", "R2", "R3"]
    params[rep] = create_parameters_improved(rep)

    u0[rep] = vcat(experimental_data[rep][1:11, 1], vitd_measured[rep][1])
    optimized_params[rep] = nothing
    
    println("$rep: $(length(params[rep])) params, u0 length=$(length(u0[rep])), VitD(t=0)=$(round(vitd_measured[rep][1], digits=1))")
end

println("\nParameters initialized with measured VitD starting values!")

In [ ]:
# CELL 6: Optimization Setup
# ============================================================================
println("\nOPTIMIZATION SETUP")
println("=" ^ 25)

tspan = (0.0, 24.0)
t_sample = [0.0, 24.0]

function solve_model(rep, p)
    try
        prob = ODEProblem((du, u, p, t) -> gene_model!(du, u, p, t, rep), u0[rep], tspan, p)
        sol = solve(prob, Tsit5(), saveat=t_sample, abstol=1e-8, reltol=1e-6)
        return length(sol.u) >= 2 ? sol : nothing
    catch
        return nothing
    end
end

function cost_function(p, rep)
    sol = solve_model(rep, p)
    if sol === nothing
        return 1e10
    end
    try
        pred = hcat([sol.u[i][1:11] for i in 1:2]...)
        exp_data = experimental_data[rep][1:11, :]
        relative_error = abs.(pred - exp_data) ./ (exp_data .+ 1.0)
        loss = sum(relative_error)
        return isnan(loss) || isinf(loss) ? 1e10 : loss
    catch e
        return 1e10
    end
end

# Test the solver
println("\nTesting solver:")
for rep in ["R1", "R2", "R3"]
    sol = solve_model(rep, params[rep])
    if sol !== nothing
        c = cost_function(params[rep], rep)
        @printf("  %s: cost=%.4f, VitD: %.1f → %.1f\n", rep, c, sol.u[1][12], sol.u[2][12])
    else
        println("  $rep: FAILED")
    end
end

In [ ]:
# CELL 7: FIXED Test Optimization Function
# ============================================================================
# The problem: BlackBoxOptim's x0 doesn't guarantee starting from those values
# Solution: Use a NARROW search range AROUND the initial parameters
# ============================================================================

function test_optimization(rep; minutes=5)
    println("\nTest optimizing $rep for $minutes minutes...")
    
    initial_p = copy(params[rep])
    initial_cost = cost_function(initial_p, rep)
    println("  Initial cost: $(round(initial_cost, digits=4))")
    
    # Create search range AROUND initial parameters
    # Allow parameters to vary by factor of 0.1x to 10x from initial values
    lower_bounds = initial_p .* 0.1
    upper_bounds = initial_p .* 10.0
    
    # Ensure minimum bounds for numerical stability
    lower_bounds = max.(lower_bounds, 1e-8)
    
    # Create the search range as vector of tuples
    search_range = [(lower_bounds[i], upper_bounds[i]) for i in 1:length(initial_p)]
    
    println("  Search range: 0.1x to 10x of initial values")
    
    result = bboptimize(p -> cost_function(p, rep);
        SearchRange = search_range,
        NumDimensions = length(initial_p),
        MaxTime = minutes * 60.0,
        PopulationSize = 100,
        TraceMode = :compact,
        TraceInterval = 30.0
    )
    
    opt_p = best_candidate(result)
    final_cost = best_fitness(result)
    
    println("  Final cost: $(round(final_cost, digits=4))")
    
    if final_cost < initial_cost
        improvement = ((initial_cost - final_cost) / initial_cost) * 100
        println("Improvement: $(round(improvement, digits=1))%")
        optimized_params[rep] = opt_p
        return opt_p, final_cost
    else
        println("No improvement - keeping initial params")
        optimized_params[rep] = initial_p
        return initial_p, initial_cost
    end
end

In [ ]:
# ============================================================================
# CELL 8: Full Optimization
# ============================================================================

function run_complete_optimization_simple(; minutes_per_rep=10)
    println("\n" * "="^60)
    println("RUNNING COMPLETE OPTIMIZATION")
    println("="^60)
    
    for rep in ["R1", "R2", "R3"]
        println("\n" * "-"^40)
        println("Optimizing $rep ($minutes_per_rep minutes)...")
        
        initial_p = copy(params[rep])
        initial_cost = cost_function(initial_p, rep)
        println("  Initial cost: $(round(initial_cost, digits=4))")
        
        # Search range around initial values
        lower_bounds = max.(initial_p .* 0.1, 1e-8)
        upper_bounds = initial_p .* 10.0
        search_range = [(lower_bounds[i], upper_bounds[i]) for i in 1:length(initial_p)]
        
        result = bboptimize(p -> cost_function(p, rep);
            SearchRange = search_range,
            NumDimensions = length(initial_p),
            MaxTime = minutes_per_rep * 60.0,
            PopulationSize = 100,
            TraceMode = :compact,
            TraceInterval = 60.0
        )
        
        opt_p = best_candidate(result)
        final_cost = best_fitness(result)
        
        println("  Final cost: $(round(final_cost, digits=4))")
        
        if final_cost < initial_cost
            improvement = ((initial_cost - final_cost) / initial_cost) * 100
            println("  Improvement: $(round(improvement, digits=1))%")
            optimized_params[rep] = opt_p
        else
            println("  No improvement - keeping initial params")
            optimized_params[rep] = initial_p
        end
    end
    
    println("\n" * "="^60)
    println("OPTIMIZATION COMPLETE!")
    println("="^60)
end

println("optimization functions loaded")

In [ ]:
# CELL 9: Results Summary Functions
# ============================================================================

function show_optimization_summary()
    println("\n" * "="^60)
    println("OPTIMIZATION SUMMARY")
    println("="^60)
    
    for rep in ["R1", "R2", "R3"]
        p = optimized_params[rep]
        if p !== nothing
            final_cost = cost_function(p, rep)
            sol = solve_model(rep, p)
            
            println("\n$rep:")
            println("  Final cost: $(round(final_cost, digits=4))")
            
            if sol !== nothing
                println("  Predictions vs Experimental:")
                for (i, gene) in enumerate(gene_names)
                    exp_t24 = experimental_data[rep][i, 2]
                    pred_t24 = sol.u[2][i]
                    err = abs(pred_t24 - exp_t24) / exp_t24 * 100
                    @printf("    %s: exp=%.0f, pred=%.0f (%.1f%% error)\n", gene, exp_t24, pred_t24, err)
                end
            end
        else
            println("\n$rep: NOT OPTIMIZED")
        end
    end
end

println("Summary function ready")

In [ ]:
# CELL 9: Additional Helper Functions
# ============================================================================
# This cell provides functions to display and save optimized parameters

# Function to show a summary of optimized parameters
function show_optimization_summary()
    println("\n" * "="^60)
    println("OPTIMIZATION SUMMARY")
    println("="^60)
    
    # Genes (first 11 have α and δ)
    genes = ["ETS2","FOS","HIF1A","EGR1","ETS1","JUN","MYC","NFKB1","STAT1","STAT3","TP53"]
    
    for rep in ["R1", "R2", "R3"]
        if optimized_params[rep] !== nothing
            p = optimized_params[rep]
            # Extract α (1:11) and δ (12:22)
            α = p[1:11]
            δ = p[12:22]
            
            println("\n$rep - COMPLETED:")
            println("  Basal rates (α): min=$(round(minimum(α), digits=6)), max=$(round(maximum(α), digits=6))")
            println("  Degradation (δ): min=$(round(minimum(δ), digits=6)), max=$(round(maximum(δ), digits=6))")
            
            # Show per-gene basal/degradation briefly
            for i in 1:11
                println("    $(genes[i]): basal=$(round(α[i], digits=6)), degradation=$(round(δ[i], digits=6))")
            end
            
            # Test the model with optimized parameters
            final_cost = cost_function(p, rep)
            println("  Final cost: $(round(final_cost, digits=6))")
            
        else
            println("\n$rep - NOT COMPLETED")
        end
    end
    
    if all(p -> p !== nothing, values(optimized_params))
        println("\n" * "="^60)
        println("ALL OPTIMIZATIONS COMPLETED SUCCESSFULLY!")
        println("Run save_equations_to_file() to save equations with optimized parameters")
        println("="^60)
    else
        missing = [rep for rep in ["R1", "R2", "R3"] if optimized_params[rep] === nothing]
        println("\nMissing optimizations: $missing")
    end
end

# Function to save all equations with optimized parameter values
function save_equations_to_file(filename="optimized_equations.txt")
    if all(p -> p !== nothing, values(optimized_params))
        open(filename, "w") do file
            write(file, "="^80 * "\n")
            write(file, "OPTIMIZED GENE REGULATORY NETWORK EQUATIONS - ALL 36 EQUATIONS\n")
            write(file, "="^80 * "\n")
            write(file, "Generated on: $(now())\n")
            write(file, "Parameter structure per replicate: α(11), δ(11), γ(71), γᵣ(46), K(71), n(71), Kᵣ(46), nᵣ(46)\n")
            write(file, "VitD parameters are FIXED (not optimized) and are taken from vitd_params.\n\n")
            
            for replicate in ["R1", "R2", "R3"]
                write(file, "\n" * "="^60 * "\n")
                write(file, "REPLICATE $replicate - OPTIMIZED EQUATIONS\n")
                write(file, "="^60 * "\n\n")
                
                p = optimized_params[replicate]
                
                # Extract parameter groups with indices matching the model
                α  = p[1:11]
                δ  = p[12:22]
                γ  = p[23:93]      # 71 activator strengths
                γr = p[94:139]     # 46 repressor strengths
                K  = p[140:210]    # 71 activator K values
                n  = p[211:281]    # 71 activator Hill coefficients
                Kr = p[282:327]    # 46 repressor K values
                nr = p[328:373]    # 46 repressor Hill coefficients
                
                # FIXED VitD parameters for this replicate
                ki = vitd_params[replicate].k
                di = vitd_params[replicate].d
                
                # Write equations with actual optimized values
                
                # 1. ETS2 (7 activators, 3 repressors)
                write(file, "1. ETS2 (7 activators, 3 repressors):\n")
                write(file, "dETS2/dt = (α1")
                write(file, " + $(round(γ[1],digits=4)) * HIF1A^$(round(n[1],digits=4)) / ($(round(K[1],digits=4))^$(round(n[1],digits=4)) + HIF1A^$(round(n[1],digits=4)))")
                write(file, " + $(round(γ[2],digits=4)) * ETS1^$(round(n[2],digits=4)) / ($(round(K[2],digits=4))^$(round(n[2],digits=4)) + ETS1^$(round(n[2],digits=4)))")
                write(file, " + $(round(γ[3],digits=4)) * JUN^$(round(n[3],digits=4)) / ($(round(K[3],digits=4))^$(round(n[3],digits=4)) + JUN^$(round(n[3],digits=4)))")
                write(file, " + $(round(γ[4],digits=4)) * MYC^$(round(n[4],digits=4)) / ($(round(K[4],digits=4))^$(round(n[4],digits=4)) + MYC^$(round(n[4],digits=4)))")
                write(file, " + $(round(γ[5],digits=4)) * STAT1^$(round(n[5],digits=4)) / ($(round(K[5],digits=4))^$(round(n[5],digits=4)) + STAT1^$(round(n[5],digits=4)))")
                write(file, " + $(round(γ[6],digits=4)) * STAT3^$(round(n[6],digits=4)) / ($(round(K[6],digits=4))^$(round(n[6],digits=4)) + STAT3^$(round(n[6],digits=4)))")
                write(file, " + $(round(γ[7],digits=4)) * TP53^$(round(n[7],digits=4)) / ($(round(K[7],digits=4))^$(round(n[7],digits=4)) + TP53^$(round(n[7],digits=4))) )")
                write(file, " * (")
                write(file, "$(round(γr[1],digits=4)) / (1 + (HIF1A/$(round(Kr[1],digits=4)))^$(round(nr[1],digits=4)))")
                write(file, " * $(round(γr[2],digits=4)) / (1 + (STAT3/$(round(Kr[2],digits=4)))^$(round(nr[2],digits=4)))")
                write(file, " * $(round(γr[3],digits=4)) / (1 + (VitD/$(round(Kr[3],digits=4)))^$(round(nr[3],digits=4)))")
                write(file, ") - $(round(δ[1],digits=4)) * ETS2\n\n")
                
                # 2. FOS (10 activators, 7 repressors)
                write(file, "2. FOS (10 activators, 7 repressors):\n")
                write(file, "dFOS/dt = (α2")
                write(file, " + $(round(γ[8],digits=4)) * ETS2^$(round(n[8],digits=4)) / ($(round(K[8],digits=4))^$(round(n[8],digits=4)) + ETS2^$(round(n[8],digits=4)))")
                write(file, " + $(round(γ[9],digits=4)) * HIF1A^$(round(n[9],digits=4)) / ($(round(K[9],digits=4))^$(round(n[9],digits=4)) + HIF1A^$(round(n[9],digits=4)))")
                write(file, " + $(round(γ[10],digits=4)) * EGR1^$(round(n[10],digits=4)) / ($(round(K[10],digits=4))^$(round(n[10],digits=4)) + EGR1^$(round(n[10],digits=4)))")
                write(file, " + $(round(γ[11],digits=4)) * ETS1^$(round(n[11],digits=4)) / ($(round(K[11],digits=4))^$(round(n[11],digits=4)) + ETS1^$(round(n[11],digits=4)))")
                write(file, " + $(round(γ[12],digits=4)) * JUN^$(round(n[12],digits=4)) / ($(round(K[12],digits=4))^$(round(n[12],digits=4)) + JUN^$(round(n[12],digits=4)))")
                write(file, " + $(round(γ[13],digits=4)) * MYC^$(round(n[13],digits=4)) / ($(round(K[13],digits=4))^$(round(n[13],digits=4)) + MYC^$(round(n[13],digits=4)))")
                write(file, " + $(round(γ[14],digits=4)) * STAT1^$(round(n[14],digits=4)) / ($(round(K[14],digits=4))^$(round(n[14],digits=4)) + STAT1^$(round(n[14],digits=4)))")
                write(file, " + $(round(γ[15],digits=4)) * STAT3^$(round(n[15],digits=4)) / ($(round(K[15],digits=4))^$(round(n[15],digits=4)) + STAT3^$(round(n[15],digits=4)))")
                write(file, " + $(round(γ[16],digits=4)) * TP53^$(round(n[16],digits=4)) / ($(round(K[16],digits=4))^$(round(n[16],digits=4)) + TP53^$(round(n[16],digits=4)))")
                write(file, " + $(round(γ[17],digits=4)) * VitD^$(round(n[17],digits=4)) / ($(round(K[17],digits=4))^$(round(n[17],digits=4)) + VitD^$(round(n[17],digits=4))) )")
                write(file, " * (")
                write(file, "$(round(γr[4],digits=4)) / (1 + (HIF1A/$(round(Kr[4],digits=4)))^$(round(nr[4],digits=4)))")
                write(file, " * $(round(γr[5],digits=4)) / (1 + (JUN/$(round(Kr[5],digits=4)))^$(round(nr[5],digits=4)))")
                write(file, " * $(round(γr[6],digits=4)) / (1 + (MYC/$(round(Kr[6],digits=4)))^$(round(nr[6],digits=4)))")
                write(file, " * $(round(γr[7],digits=4)) / (1 + (STAT1/$(round(Kr[7],digits=4)))^$(round(nr[7],digits=4)))")
                write(file, " * $(round(γr[8],digits=4)) / (1 + (STAT3/$(round(Kr[8],digits=4)))^$(round(nr[8],digits=4)))")
                write(file, " * $(round(γr[9],digits=4)) / (1 + (TP53/$(round(Kr[9],digits=4)))^$(round(nr[9],digits=4)))")
                write(file, " * $(round(γr[10],digits=4)) / (1 + (VitD/$(round(Kr[10],digits=4)))^$(round(nr[10],digits=4)))")
                write(file, ") - $(round(δ[2],digits=4)) * FOS\n\n")
                
                # 3. HIF1A (5 activators, 1 repressor)
                write(file, "3. HIF1A (5 activators, 1 repressor):\n")
                write(file, "dHIF1A/dt = (α3")
                write(file, " + $(round(γ[18],digits=4)) * EGR1^$(round(n[18],digits=4)) / ($(round(K[18],digits=4))^$(round(n[18],digits=4)) + EGR1^$(round(n[18],digits=4)))")
                write(file, " + $(round(γ[19],digits=4)) * MYC^$(round(n[19],digits=4)) / ($(round(K[19],digits=4))^$(round(n[19],digits=4)) + MYC^$(round(n[19],digits=4)))")
                write(file, " + $(round(γ[20],digits=4)) * NFKB1^$(round(n[20],digits=4)) / ($(round(K[20],digits=4))^$(round(n[20],digits=4)) + NFKB1^$(round(n[20],digits=4)))")
                write(file, " + $(round(γ[21],digits=4)) * STAT3^$(round(n[21],digits=4)) / ($(round(K[21],digits=4))^$(round(n[21],digits=4)) + STAT3^$(round(n[21],digits=4)))")
                write(file, " + $(round(γ[22],digits=4)) * VitD^$(round(n[22],digits=4)) / ($(round(K[22],digits=4))^$(round(n[22],digits=4)) + VitD^$(round(n[22],digits=4))) )")
                write(file, " * (")
                write(file, "$(round(γr[11],digits=4)) / (1 + (VitD/$(round(Kr[11],digits=4)))^$(round(nr[11],digits=4)))")
                write(file, ") - $(round(δ[3],digits=4)) * HIF1A\n\n")
                
                # 4. EGR1 (8 activators, 3 repressors)
                write(file, "4. EGR1 (8 activators, 3 repressors):\n")
                write(file, "dEGR1/dt = (α4")
                write(file, " + $(round(γ[23],digits=4)) * ETS2^$(round(n[23],digits=4)) / ($(round(K[23],digits=4))^$(round(n[23],digits=4)) + ETS2^$(round(n[23],digits=4)))")
                write(file, " + $(round(γ[24],digits=4)) * HIF1A^$(round(n[24],digits=4)) / ($(round(K[24],digits=4))^$(round(n[24],digits=4)) + HIF1A^$(round(n[24],digits=4)))")
                write(file, " + $(round(γ[25],digits=4)) * ETS1^$(round(n[25],digits=4)) / ($(round(K[25],digits=4))^$(round(n[25],digits=4)) + ETS1^$(round(n[25],digits=4)))")
                write(file, " + $(round(γ[26],digits=4)) * MYC^$(round(n[26],digits=4)) / ($(round(K[26],digits=4))^$(round(n[26],digits=4)) + MYC^$(round(n[26],digits=4)))")
                write(file, " + $(round(γ[27],digits=4)) * STAT1^$(round(n[27],digits=4)) / ($(round(K[27],digits=4))^$(round(n[27],digits=4)) + STAT1^$(round(n[27],digits=4)))")
                write(file, " + $(round(γ[28],digits=4)) * STAT3^$(round(n[28],digits=4)) / ($(round(K[28],digits=4))^$(round(n[28],digits=4)) + STAT3^$(round(n[28],digits=4)))")
                write(file, " + $(round(γ[29],digits=4)) * TP53^$(round(n[29],digits=4)) / ($(round(K[29],digits=4))^$(round(n[29],digits=4)) + TP53^$(round(n[29],digits=4)))")
                write(file, " + $(round(γ[30],digits=4)) * VitD^$(round(n[30],digits=4)) / ($(round(K[30],digits=4))^$(round(n[30],digits=4)) + VitD^$(round(n[30],digits=4))) )")
                write(file, " * (")
                write(file, "$(round(γr[12],digits=4)) / (1 + (HIF1A/$(round(Kr[12],digits=4)))^$(round(nr[12],digits=4)))")
                write(file, " * $(round(γr[13],digits=4)) / (1 + (STAT3/$(round(Kr[13],digits=4)))^$(round(nr[13],digits=4)))")
                write(file, " * $(round(γr[14],digits=4)) / (1 + (VitD/$(round(Kr[14],digits=4)))^$(round(nr[14],digits=4)))")
                write(file, ") - $(round(δ[4],digits=4)) * EGR1\n\n")
                
                # 5. ETS1 (4 activators, 8 repressors)
                write(file, "5. ETS1 (4 activators, 8 repressors):\n")
                write(file, "dETS1/dt = (α5")
                write(file, " + $(round(γ[31],digits=4)) * HIF1A^$(round(n[31],digits=4)) / ($(round(K[31],digits=4))^$(round(n[31],digits=4)) + HIF1A^$(round(n[31],digits=4)))")
                write(file, " + $(round(γ[32],digits=4)) * JUN^$(round(n[32],digits=4)) / ($(round(K[32],digits=4))^$(round(n[32],digits=4)) + JUN^$(round(n[32],digits=4)))")
                write(file, " + $(round(γ[33],digits=4)) * STAT3^$(round(n[33],digits=4)) / ($(round(K[33],digits=4))^$(round(n[33],digits=4)) + STAT3^$(round(n[33],digits=4)))")
                write(file, " + $(round(γ[34],digits=4)) * VitD^$(round(n[34],digits=4)) / ($(round(K[34],digits=4))^$(round(n[34],digits=4)) + VitD^$(round(n[34],digits=4))) )")
                write(file, " * (")
                write(file, "$(round(γr[15],digits=4)) / (1 + (FOS/$(round(Kr[15],digits=4)))^$(round(nr[15],digits=4)))")
                write(file, " * $(round(γr[16],digits=4)) / (1 + (HIF1A/$(round(Kr[16],digits=4)))^$(round(nr[16],digits=4)))")
                write(file, " * $(round(γr[17],digits=4)) / (1 + (JUN/$(round(Kr[17],digits=4)))^$(round(nr[17],digits=4)))")
                write(file, " * $(round(γr[18],digits=4)) / (1 + (MYC/$(round(Kr[18],digits=4)))^$(round(nr[18],digits=4)))")
                write(file, " * $(round(γr[19],digits=4)) / (1 + (STAT1/$(round(Kr[19],digits=4)))^$(round(nr[19],digits=4)))")
                write(file, " * $(round(γr[20],digits=4)) / (1 + (STAT3/$(round(Kr[20],digits=4)))^$(round(nr[20],digits=4)))")
                write(file, " * $(round(γr[21],digits=4)) / (1 + (TP53/$(round(Kr[21],digits=4)))^$(round(nr[21],digits=4)))")
                write(file, " * $(round(γr[22],digits=4)) / (1 + (VitD/$(round(Kr[22],digits=4)))^$(round(nr[22],digits=4)))")
                write(file, ") - $(round(δ[5],digits=4)) * ETS1\n\n")
                
                # 6. JUN (7 activators, 7 repressors)
                write(file, "6. JUN (7 activators, 7 repressors):\n")
                write(file, "dJUN/dt = (α6")
                write(file, " + $(round(γ[35],digits=4)) * HIF1A^$(round(n[35],digits=4)) / ($(round(K[35],digits=4))^$(round(n[35],digits=4)) + HIF1A^$(round(n[35],digits=4)))")
                write(file, " + $(round(γ[36],digits=4)) * EGR1^$(round(n[36],digits=4)) / ($(round(K[36],digits=4))^$(round(n[36],digits=4)) + EGR1^$(round(n[36],digits=4)))")
                write(file, " + $(round(γ[37],digits=4)) * MYC^$(round(n[37],digits=4)) / ($(round(K[37],digits=4))^$(round(n[37],digits=4)) + MYC^$(round(n[37],digits=4)))")
                write(file, " + $(round(γ[38],digits=4)) * STAT1^$(round(n[38],digits=4)) / ($(round(K[38],digits=4))^$(round(n[38],digits=4)) + STAT1^$(round(n[38],digits=4)))")
                write(file, " + $(round(γ[39],digits=4)) * STAT3^$(round(n[39],digits=4)) / ($(round(K[39],digits=4))^$(round(n[39],digits=4)) + STAT3^$(round(n[39],digits=4)))")
                write(file, " + $(round(γ[40],digits=4)) * TP53^$(round(n[40],digits=4)) / ($(round(K[40],digits=4))^$(round(n[40],digits=4)) + TP53^$(round(n[40],digits=4)))")
                write(file, " + $(round(γ[41],digits=4)) * VitD^$(round(n[41],digits=4)) / ($(round(K[41],digits=4))^$(round(n[41],digits=4)) + VitD^$(round(n[41],digits=4))) )")
                write(file, " * (")
                write(file, "$(round(γr[23],digits=4)) / (1 + (FOS/$(round(Kr[23],digits=4)))^$(round(nr[23],digits=4)))")
                write(file, " * $(round(γr[24],digits=4)) / (1 + (HIF1A/$(round(Kr[24],digits=4)))^$(round(nr[24],digits=4)))")
                write(file, " * $(round(γr[25],digits=4)) / (1 + (MYC/$(round(Kr[25],digits=4)))^$(round(nr[25],digits=4)))")
                write(file, " * $(round(γr[26],digits=4)) / (1 + (STAT1/$(round(Kr[26],digits=4)))^$(round(nr[26],digits=4)))")
                write(file, " * $(round(γr[27],digits=4)) / (1 + (STAT3/$(round(Kr[27],digits=4)))^$(round(nr[27],digits=4)))")
                write(file, " * $(round(γr[28],digits=4)) / (1 + (TP53/$(round(Kr[28],digits=4)))^$(round(nr[28],digits=4)))")
                write(file, " * $(round(γr[29],digits=4)) / (1 + (VitD/$(round(Kr[29],digits=4)))^$(round(nr[29],digits=4)))")
                write(file, ") - $(round(δ[6],digits=4)) * JUN\n\n")
                
                # 7. MYC (8 activators, 6 repressors)
                write(file, "7. MYC (8 activators, 6 repressors):\n")
                write(file, "dMYC/dt = (α7")
                write(file, " + $(round(γ[42],digits=4)) * ETS2^$(round(n[42],digits=4)) / ($(round(K[42],digits=4))^$(round(n[42],digits=4)) + ETS2^$(round(n[42],digits=4)))")
                write(file, " + $(round(γ[43],digits=4)) * HIF1A^$(round(n[43],digits=4)) / ($(round(K[43],digits=4))^$(round(n[43],digits=4)) + HIF1A^$(round(n[43],digits=4)))")
                write(file, " + $(round(γ[44],digits=4)) * EGR1^$(round(n[44],digits=4)) / ($(round(K[44],digits=4))^$(round(n[44],digits=4)) + EGR1^$(round(n[44],digits=4)))")
                write(file, " + $(round(γ[45],digits=4)) * NFKB1^$(round(n[45],digits=4)) / ($(round(K[45],digits=4))^$(round(n[45],digits=4)) + NFKB1^$(round(n[45],digits=4)))")
                write(file, " + $(round(γ[46],digits=4)) * STAT1^$(round(n[46],digits=4)) / ($(round(K[46],digits=4))^$(round(n[46],digits=4)) + STAT1^$(round(n[46],digits=4)))")
                write(file, " + $(round(γ[47],digits=4)) * STAT3^$(round(n[47],digits=4)) / ($(round(K[47],digits=4))^$(round(n[47],digits=4)) + STAT3^$(round(n[47],digits=4)))")
                write(file, " + $(round(γ[48],digits=4)) * TP53^$(round(n[48],digits=4)) / ($(round(K[48],digits=4))^$(round(n[48],digits=4)) + TP53^$(round(n[48],digits=4)))")
                write(file, " + $(round(γ[49],digits=4)) * VitD^$(round(n[49],digits=4)) / ($(round(K[49],digits=4))^$(round(n[49],digits=4)) + VitD^$(round(n[49],digits=4))) )")
                write(file, " * (")
                write(file, "$(round(γr[30],digits=4)) / (1 + (FOS/$(round(Kr[30],digits=4)))^$(round(nr[30],digits=4)))")
                write(file, " * $(round(γr[31],digits=4)) / (1 + (HIF1A/$(round(Kr[31],digits=4)))^$(round(nr[31],digits=4)))")
                write(file, " * $(round(γr[32],digits=4)) / (1 + (JUN/$(round(Kr[32],digits=4)))^$(round(nr[32],digits=4)))")
                write(file, " * $(round(γr[33],digits=4)) / (1 + (STAT3/$(round(Kr[33],digits=4)))^$(round(nr[33],digits=4)))")
                write(file, " * $(round(γr[34],digits=4)) / (1 + (TP53/$(round(Kr[34],digits=4)))^$(round(nr[34],digits=4)))")
                write(file, " * $(round(γr[35],digits=4)) / (1 + (VitD/$(round(Kr[35],digits=4)))^$(round(nr[35],digits=4)))")
                write(file, ") - $(round(δ[7],digits=4)) * MYC\n\n")
                
                # 8. NFKB1 (7 activators, 4 repressors)
                write(file, "8. NFKB1 (7 activators, 4 repressors):\n")
                write(file, "dNFKB1/dt = (α8")
                write(file, " + $(round(γ[50],digits=4)) * HIF1A^$(round(n[50],digits=4)) / ($(round(K[50],digits=4))^$(round(n[50],digits=4)) + HIF1A^$(round(n[50],digits=4)))")
                write(file, " + $(round(γ[51],digits=4)) * EGR1^$(round(n[51],digits=4)) / ($(round(K[51],digits=4))^$(round(n[51],digits=4)) + EGR1^$(round(n[51],digits=4)))")
                write(file, " + $(round(γ[52],digits=4)) * ETS1^$(round(n[52],digits=4)) / ($(round(K[52],digits=4))^$(round(n[52],digits=4)) + ETS1^$(round(n[52],digits=4)))")
                write(file, " + $(round(γ[53],digits=4)) * MYC^$(round(n[53],digits=4)) / ($(round(K[53],digits=4))^$(round(n[53],digits=4)) + MYC^$(round(n[53],digits=4)))")
                write(file, " + $(round(γ[54],digits=4)) * STAT1^$(round(n[54],digits=4)) / ($(round(K[54],digits=4))^$(round(n[54],digits=4)) + STAT1^$(round(n[54],digits=4)))")
                write(file, " + $(round(γ[55],digits=4)) * STAT3^$(round(n[55],digits=4)) / ($(round(K[55],digits=4))^$(round(n[55],digits=4)) + STAT3^$(round(n[55],digits=4)))")
                write(file, " + $(round(γ[56],digits=4)) * TP53^$(round(n[56],digits=4)) / ($(round(K[56],digits=4))^$(round(n[56],digits=4)) + TP53^$(round(n[56],digits=4))) )")
                write(file, " * (")
                write(file, "$(round(γr[36],digits=4)) / (1 + (FOS/$(round(Kr[36],digits=4)))^$(round(nr[36],digits=4)))")
                write(file, " * $(round(γr[37],digits=4)) / (1 + (HIF1A/$(round(Kr[37],digits=4)))^$(round(nr[37],digits=4)))")
                write(file, " * $(round(γr[38],digits=4)) / (1 + (STAT3/$(round(Kr[38],digits=4)))^$(round(nr[38],digits=4)))")
                write(file, " * $(round(γr[39],digits=4)) / (1 + (VitD/$(round(Kr[39],digits=4)))^$(round(nr[39],digits=4)))")
                write(file, ") - $(round(δ[8],digits=4)) * NFKB1\n\n")
                
                # 9. STAT1 (3 activators, 0 repressors)
                write(file, "9. STAT1 (3 activators, 0 repressors):\n")
                write(file, "dSTAT1/dt = (α9")
                write(file, " + $(round(γ[57],digits=4)) * STAT3^$(round(n[57],digits=4)) / ($(round(K[57],digits=4))^$(round(n[57],digits=4)) + STAT3^$(round(n[57],digits=4)))")
                write(file, " + $(round(γ[58],digits=4)) * TP53^$(round(n[58],digits=4)) / ($(round(K[58],digits=4))^$(round(n[58],digits=4)) + TP53^$(round(n[58],digits=4)))")
                write(file, " + $(round(γ[59],digits=4)) * VitD^$(round(n[59],digits=4)) / ($(round(K[59],digits=4))^$(round(n[59],digits=4)) + VitD^$(round(n[59],digits=4))) )")
                write(file, " - $(round(δ[9],digits=4)) * STAT1\n\n")
                
                # 10. STAT3 (2 activators, 1 repressor)
                write(file, "10. STAT3 (2 activators, 1 repressor):\n")
                write(file, "dSTAT3/dt = (α10")
                write(file, " + $(round(γ[60],digits=4)) * STAT1^$(round(n[60],digits=4)) / ($(round(K[60],digits=4))^$(round(n[60],digits=4)) + STAT1^$(round(n[60],digits=4)))")
                write(file, " + $(round(γ[61],digits=4)) * VitD^$(round(n[61],digits=4)) / ($(round(K[61],digits=4))^$(round(n[61],digits=4)) + VitD^$(round(n[61],digits=4))) )")
                write(file, " * (")
                write(file, "$(round(γr[40],digits=4)) / (1 + (TP53/$(round(Kr[40],digits=4)))^$(round(nr[40],digits=4)))")
                write(file, ") - $(round(δ[10],digits=4)) * STAT3\n\n")
                
                # 11. TP53 (10 activators, 6 repressors)
                write(file, "11. TP53 (10 activators, 6 repressors):\n")
                write(file, "dTP53/dt = (α11")
                write(file, " + $(round(γ[62],digits=4)) * ETS2^$(round(n[62],digits=4)) / ($(round(K[62],digits=4))^$(round(n[62],digits=4)) + ETS2^$(round(n[62],digits=4)))")
                write(file, " + $(round(γ[63],digits=4)) * FOS^$(round(n[63],digits=4)) / ($(round(K[63],digits=4))^$(round(n[63],digits=4)) + FOS^$(round(n[63],digits=4)))")
                write(file, " + $(round(γ[64],digits=4)) * HIF1A^$(round(n[64],digits=4)) / ($(round(K[64],digits=4))^$(round(n[64],digits=4)) + HIF1A^$(round(n[64],digits=4)))")
                write(file, " + $(round(γ[65],digits=4)) * EGR1^$(round(n[65],digits=4)) / ($(round(K[65],digits=4))^$(round(n[65],digits=4)) + EGR1^$(round(n[65],digits=4)))")
                write(file, " + $(round(γ[66],digits=4)) * ETS1^$(round(n[66],digits=4)) / ($(round(K[66],digits=4))^$(round(n[66],digits=4)) + ETS1^$(round(n[66],digits=4)))")
                write(file, " + $(round(γ[67],digits=4)) * MYC^$(round(n[67],digits=4)) / ($(round(K[67],digits=4))^$(round(n[67],digits=4)) + MYC^$(round(n[67],digits=4)))")
                write(file, " + $(round(γ[68],digits=4)) * NFKB1^$(round(n[68],digits=4)) / ($(round(K[68],digits=4))^$(round(n[68],digits=4)) + NFKB1^$(round(n[68],digits=4)))")
                write(file, " + $(round(γ[69],digits=4)) * STAT1^$(round(n[69],digits=4)) / ($(round(K[69],digits=4))^$(round(n[69],digits=4)) + STAT1^$(round(n[69],digits=4)))")
                write(file, " + $(round(γ[70],digits=4)) * STAT3^$(round(n[70],digits=4)) / ($(round(K[70],digits=4))^$(round(n[70],digits=4)) + STAT3^$(round(n[70],digits=4)))")
                write(file, " + $(round(γ[71],digits=4)) * VitD^$(round(n[71],digits=4)) / ($(round(K[71],digits=4))^$(round(n[71],digits=4)) + VitD^$(round(n[71],digits=4))) )")
                write(file, " * (")
                write(file, "$(round(γr[41],digits=4)) / (1 + (HIF1A/$(round(Kr[41],digits=4)))^$(round(nr[41],digits=4)))")
                write(file, " * $(round(γr[42],digits=4)) / (1 + (JUN/$(round(Kr[42],digits=4)))^$(round(nr[42],digits=4)))")
                write(file, " * $(round(γr[43],digits=4)) / (1 + (MYC/$(round(Kr[43],digits=4)))^$(round(nr[43],digits=4)))")
                write(file, " * $(round(γr[44],digits=4)) / (1 + (STAT1/$(round(Kr[44],digits=4)))^$(round(nr[44],digits=4)))")
                write(file, " * $(round(γr[45],digits=4)) / (1 + (STAT3/$(round(Kr[45],digits=4)))^$(round(nr[45],digits=4)))")
                write(file, " * $(round(γr[46],digits=4)) / (1 + (VitD/$(round(Kr[46],digits=4)))^$(round(nr[46],digits=4)))")
                write(file, ") - $(round(δ[11],digits=4)) * TP53\n\n")
                
                # 12. VitD (FIXED)
                write(file, "12. VitD (FIXED PARAMETERS - NOT OPTIMIZED):\n")
                write(file, "dVitD/dt = $(round(ki,digits=4)) - $(round(di,digits=6)) * VitD\n\n")
                
                # Final cost for this replicate
                final_cost = cost_function(p, replicate)
                write(file, "Final fitting cost for $replicate: $(round(final_cost, digits=6))\n")
                write(file, "\n" * "-"^60 * "\n")
            end
            
            write(file, "\n" * "="^80 * "\n")
            write(file, "SUMMARY:\n")
            write(file, "- 36 optimized equations (12 variables × 3 replicates; VitD fixed per replicate)\n")
            write(file, "- Parameterization per replicate: 373 optimized parameters (11 α + 11 δ + 71 γ + 46 γᵣ + 71 K + 71 n + 46 Kᵣ + 46 nᵣ)\n")
            write(file, "- VitD parameters are FIXED (from vitd_params) and not part of optimization vector p\n")
            write(file, "- All regulatory interactions from the updated model are preserved with actual optimized values\n")
            write(file, "="^80 * "\n")
        end
        
        println("All 36 equations with optimized parameters saved to: $filename")
        println("File contains complete equations with actual parameter values for all replicates.")
        
    else
        println("Cannot save equations - optimization not completed for all replicates")
        println("Missing: ", [rep for rep in ["R1", "R2", "R3"] if optimized_params[rep] === nothing])
    end
end

# Function to show concise summary of parameters
function show_final_equations()
    if all(p -> p !== nothing, values(optimized_params))
        genes = ["ETS2","FOS","HIF1A","EGR1","ETS1","JUN","MYC","NFKB1","STAT1","STAT3","TP53"]
        
        for rep in ["R1", "R2", "R3"]
            println("\n" * "="^60)
            println("$rep - FINAL OPTIMIZED PARAMETERS (concise)")
            println("="^60)
            
            p = optimized_params[rep]
            α, δ = p[1:11], p[12:22]
            
            for i in 1:11
                println(rpad("$(i). $(genes[i]):", 18, " "), " basal=$(round(α[i],digits=6)), degradation=$(round(δ[i],digits=6))")
            end
            
            ki = vitd_params[rep].k
            di = vitd_params[rep].d
            println(rpad("12. VitD:", 18, " "), " synthesis=$(round(ki,digits=6)), degradation=$(round(di,digits=6)) [FIXED]")
            
            final_cost = cost_function(p, rep)
            println("Final fitting cost: $(round(final_cost, digits=6))")
        end
    else
        println("Optimization not completed for all replicates")
        println("Missing: ", [rep for rep in ["R1", "R2", "R3"] if optimized_params[rep] === nothing])
    end
end

println("Helper functions loaded successfully!")
println("Available functions:")
println("  - show_optimization_summary() : Display optimization results summary")
println("  - save_equations_to_file([filename]) : Save all equations with optimized parameters")
println("  - show_final_equations() : Display concise parameter summary")

In [ ]:
# CELL 10: Plotting Functions
# ============================================================================

function get_simulation_results()
    results = Dict()
    for rep in ["R1", "R2", "R3"]
        p = optimized_params[rep]
        if p !== nothing
            sol = solve_model(rep, p)
            if sol !== nothing
                results[rep] = hcat([sol.u[i][1:11] for i in 1:2]...)
            end
        end
    end
    return results
end

function get_experimental_distributions()
    distributions = Dict()
    for rep in ["R1", "R2", "R3"]
        distributions[rep] = Dict()
        for (i, gene) in enumerate(gene_names)
            gene_row = filter(row -> row.hgnc_symbol == gene, raw_data)
            if nrow(gene_row) > 0
                row = gene_row[1, :]
                t0_cols = filter(col -> startswith(string(col), "$(rep)_0_"), names(raw_data))
                t0_values = [convert_european_decimal(row[col]) for col in t0_cols if !ismissing(row[col])]
                t0_values = filter(v -> !isnan(v) && v > 0, t0_values)
                t24_cols = filter(col -> startswith(string(col), "$(rep)_24_"), names(raw_data))
                t24_values = [convert_european_decimal(row[col]) for col in t24_cols if !ismissing(row[col])]
                t24_values = filter(v -> !isnan(v) && v > 0, t24_values)
                distributions[rep][gene] = Dict("t0" => t0_values, "t24" => t24_values)
            end
        end
    end
    return distributions
end

function create_all_plots()
    if !isdir("Gene_model_plots")
        mkdir("Gene_model_plots")
    end
    
    sim_results = get_simulation_results()
    exp_distributions = get_experimental_distributions()
    
    if isempty(sim_results)
        println("No results. Run optimization first.")
        return
    end
    
    for rep in ["R1", "R2", "R3"]
        if haskey(sim_results, rep)
            for (i, gene) in enumerate(gene_names)
                sim_t0 = sim_results[rep][i, 1]
                sim_t24 = sim_results[rep][i, 2]
                exp_t0 = exp_distributions[rep][gene]["t0"]
                exp_t24 = exp_distributions[rep][gene]["t24"]
                exp_t0_mean = mean(exp_t0)
                exp_t24_mean = mean(exp_t24)
                
                p = plot(size=(600, 500), dpi=300, grid=false, framestyle=:axes)
                boxplot!(p, [1], [exp_t0], color=:lightblue, alpha=0.7, bar_width=0.25, label="")
                boxplot!(p, [2], [exp_t24], color=:lightcoral, alpha=0.7, bar_width=0.25, label="")
                plot!(p, [0.85, 1.15], [exp_t0_mean, exp_t0_mean], color=:blue, linewidth=4, label="")
                plot!(p, [1.85, 2.15], [exp_t24_mean, exp_t24_mean], color=:red, linewidth=4, label="")
                scatter!(p, [1], [sim_t0], markersize=8, color=:darkblue, marker=:diamond, label="Sim t=0")
                scatter!(p, [2], [sim_t24], markersize=8, color=:darkred, marker=:diamond, label="Sim t=24")
                xlabel!(p, "Time Point")
                ylabel!(p, "Expression (CPM)")
                title!(p, "$gene: Exp vs Sim ($rep)")
                plot!(p, xticks=([1, 2], ["0h", "24h"]), xlims=(0.3, 2.7))
                
                savefig(p, "Gene_model_plots/$(gene)_$(rep).png")
            end
            println("Saved plots for $rep")
        end
    end
    println("\nAll plots saved to Gene_model_plots/")
end

println("Plotting functions ready")
println("Run: create_all_plots()")

In [ ]:
# CELL 11: Run Everything
# ============================================================================

println("\n" * "="^60)
println("READY TO RUN")
println("="^60)
println("\n1. Quick test (1 min each):")
println("   test_optimization(\"R1\"; minutes=1)")
println("\n2. Full optimization (10 min each):")
println("   run_complete_optimization_simple(minutes_per_rep=10)")
println("\n3. View results:")
println("   show_optimization_summary()")
println("\n4. Create plots:")
println("   create_all_plots()")

In [ ]:
# Quick test all replicates (1 minute each)
#test_optimization("R1"; minutes=1)
#test_optimization("R2"; minutes=1)
#test_optimization("R3"; minutes=1)

In [ ]:
# Full optimization (10 minutes each)
run_complete_optimization_simple(minutes_per_rep=3)

In [ ]:
# View results and create plots
show_optimization_summary()
save_equations_to_file("optimized_equations_model_1.txt")
#create_all_plots()

In [ ]:
# ================================================================================
# COMPLETE FIREPROOF VALIDATION - CORRECTED BOXPLOTS
# ================================================================================

println("\n" * "="^80)
println(" FIREPROOF VALIDATION - Inter-bolus Dynamics (COMPLETE)")
println("="^80)

using DifferentialEquations, Statistics, Printf, CSV, DataFrames, Plots, StatsPlots
gr()

# Create output directory
if !isdir("Fireproof_Validation1")
    mkdir("Fireproof_Validation1")
end

# Load validation data
validation_file = "validation_data_all_12_common_individuals.csv"
if !isfile(validation_file)
    println(" Validation file not found: $validation_file")
    println("Skipping fireproof validation")
else
    validation_raw = CSV.read(validation_file, DataFrame; delim=';', header=true)
    println(" Loaded validation data: $(nrow(validation_raw)) rows")

    # ============================================================================
    # VitD DECAY PARAMETERS - IN DAYS (matching original working script)
    # ============================================================================
    vitd_validation = Dict(
        "R1" => 0.013377,  # d = 0.013377 per day (half-life = 51.8 days)
        "R2" => 0.002917,  # d = 0.002917 per day (half-life = 237.6 days)  
        "R3" => 0.003129   # d = 0.003129 per day (half-life = 221.5 days)
    )
    
    println("\n VitD Decay Rates (per DAY):")
    for rep in ["R1", "R2", "R3"]
        half_life_days = log(2) / vitd_validation[rep]
        @printf("  %s: %.6f/day → t½ = %.1f days\n", 
                rep, vitd_validation[rep], half_life_days)
    end

    # ============================================================================
    # HELPER FUNCTIONS
    # ============================================================================
    
    function get_validation_data(gene, timepoint)
        """Extract validation data for a specific gene and timepoint"""
        gene_row = filter(r -> r.hgnc_symbol == gene, validation_raw)
        if nrow(gene_row) > 0
            row = gene_row[1, :]
            cols = filter(c -> startswith(string(c), "$(timepoint)_"), names(validation_raw))
            vals = Float64[]
            for c in cols
                try
                    v = convert_european_decimal(row[c])
                    if !isnan(v) && v > 0
                        push!(vals, v)
                    end
                catch
                end
            end
            if length(vals) > 0
                return (mean=mean(vals), std=length(vals)>1 ? std(vals) : 0.0, values=vals)
            end
        end
        return (mean=NaN, std=0.0, values=Float64[])
    end

    # ============================================================================
    # VALIDATION MODEL - TIME IN DAYS
    # ============================================================================
    
    function gene_model_validation!(du, u, p, t, rep)
        """
        Modified gene model for inter-bolus period:
        - Genes follow normal dynamics (optimized parameters)
        - VitD decays exponentially (no synthesis, k=0)
        - Time is in DAYS (matching original working script)
        """
        # Use your existing model for all genes
        gene_model!(du, u, p, t, rep)
        
        # Override VitD equation: decay only (k=0)
        # Using DAYS as time unit
        VitD = max(u[12], 1e-6)
        di = vitd_validation[rep]
        du[12] = 0.0 - di * VitD
        
        nothing
    end

    # ============================================================================
    # RUN FIREPROOF TESTS
    # ============================================================================
    
    function run_fireproof_test(start_tp, end_tp, rep, duration_days)
        println("\n" * "="^60)
        println(" Fireproof Test: $start_tp → $end_tp")
        println("   Replicate: $rep")
        println("   Duration: $duration_days days")
        println("="^60)
        
        # ====================================================================
        # STEP 1: Get initial conditions from experimental data
        # ====================================================================
        u0_test = zeros(12)
        
        # Get INITIAL VALUES (start_tp)
        initial_vals = Dict{String, Vector{Float64}}()
        for (i, gene) in enumerate(gene_names)
            if start_tp == "d1"
                # Use R1 experimental at t=24h
                u0_test[i] = experimental_data["R1"][i, 2]
                # For boxplot, get individual values
                gene_row = filter(r -> r.hgnc_symbol == gene, raw_data)
                if nrow(gene_row) > 0
                    row = gene_row[1, :]
                    t24_cols = filter(c -> startswith(string(c), "R1_24_"), names(raw_data))
                    t24_vals = [convert_european_decimal(row[c]) for c in t24_cols if !ismissing(row[c])]
                    t24_vals = filter(v -> !isnan(v) && v > 0, t24_vals)
                    initial_vals[gene] = t24_vals
                else
                    initial_vals[gene] = [u0_test[i]]
                end
            elseif start_tp == "d29"
                # Use R2 experimental at t=24h
                u0_test[i] = experimental_data["R2"][i, 2]
                gene_row = filter(r -> r.hgnc_symbol == gene, raw_data)
                if nrow(gene_row) > 0
                    row = gene_row[1, :]
                    t24_cols = filter(c -> startswith(string(c), "R2_24_"), names(raw_data))
                    t24_vals = [convert_european_decimal(row[c]) for c in t24_cols if !ismissing(row[c])]
                    t24_vals = filter(v -> !isnan(v) && v > 0, t24_vals)
                    initial_vals[gene] = t24_vals
                else
                    initial_vals[gene] = [u0_test[i]]
                end
            elseif start_tp == "d57"
                # Use R3 experimental at t=24h
                u0_test[i] = experimental_data["R3"][i, 2]
                gene_row = filter(r -> r.hgnc_symbol == gene, raw_data)
                if nrow(gene_row) > 0
                    row = gene_row[1, :]
                    t24_cols = filter(c -> startswith(string(c), "R3_24_"), names(raw_data))
                    t24_vals = [convert_european_decimal(row[c]) for c in t24_cols if !ismissing(row[c])]
                    t24_vals = filter(v -> !isnan(v) && v > 0, t24_vals)
                    initial_vals[gene] = t24_vals
                else
                    initial_vals[gene] = [u0_test[i]]
                end
            else
                data = get_validation_data(gene, start_tp)
                u0_test[i] = isnan(data.mean) ? 1000.0 : data.mean
                initial_vals[gene] = data.values
            end
        end
        
        # VitD initial - normalized value
        u0_test[12] = 1.0
        
        println("\n Initial Conditions:")
        for (i, gene) in enumerate(gene_names)
            @printf("  %-10s: %.1f\n", gene, u0_test[i])
        end
        @printf("  %-10s: %.3f (normalized)\n", "VitD", u0_test[12])
        
        # ====================================================================
        # STEP 2: Get target values
        # ====================================================================
        targets = zeros(11)
        target_stds = zeros(11)
        target_vals = Dict{String, Vector{Float64}}()
        
        println("\n Target Values (from $end_tp validation data):")
        for (i, gene) in enumerate(gene_names)
            data = get_validation_data(gene, end_tp)
            targets[i] = data.mean
            target_stds[i] = data.std
            target_vals[gene] = data.values
            
            if !isnan(data.mean)
                @printf("  %-10s: %.1f ± %.1f (n=%d)\n", 
                        gene, data.mean, data.std, length(data.values))
            end
        end
        
        # ====================================================================
        # STEP 3: Simulate - TIME IN DAYS
        # ====================================================================
        p_opt = optimized_params[rep]
        if p_opt === nothing
            println("\n No optimized parameters for $rep")
            return nothing
        end
        
        # CRITICAL: tspan in DAYS (not hours)
        tspan = (0.0, Float64(duration_days))
        
        println("\n Simulation Settings:")
        println("  Duration: $duration_days days")
        println("  VitD decay rate: $(vitd_validation[rep]) per day")
        println("  Time unit: DAYS")
        
        try
            prob = ODEProblem(
                (du, u, p, t) -> gene_model_validation!(du, u, p, t, rep),
                u0_test,
                tspan,
                p_opt
            )
            
            # Solve - save daily snapshots
            t_save = collect(0.0:1.0:Float64(duration_days))
            sol = solve(prob, Tsit5(), 
                       saveat=t_save,
                       abstol=1e-8, 
                       reltol=1e-6)
            
            if length(sol.u) == 0
                println("\n Solver failed")
                return nothing
            end
            
            # ================================================================
            # STEP 4: Extract results
            # ================================================================
            predicted = sol.u[end][1:11]
            vitd_initial = sol.u[1][12]
            vitd_final = sol.u[end][12]
            vitd_retention = 100.0 * vitd_final / vitd_initial
            
            # Store trajectory data
            gene_trajectories = Dict{String, Vector{Float64}}()
            for (i, gene) in enumerate(gene_names)
                gene_trajectories[gene] = [u[i] for u in sol.u]
            end
            vitd_trajectory = [u[12] for u in sol.u]
            
            # ================================================================
            # STEP 5: Calculate errors
            # ================================================================
            errors = Float64[]
            valid_genes = String[]
            
            println("\n" * "="^60)
            println("RESULTS")
            println("="^60)
            println("\nGene       | Initial  | Predicted | Target   | Error    | Status")
            println("-"^70)
            
            for (i, gene) in enumerate(gene_names)
                if !isnan(targets[i]) && targets[i] > 0
                    initial_val = u0_test[i]
                    pred_val = predicted[i]
                    target_val = targets[i]
                    
                    rel_error = abs(pred_val - target_val) / (target_val + 1.0)
                    push!(errors, rel_error)
                    push!(valid_genes, gene)
                    
                    status = rel_error < 0.3 ? "EXCELLENT" : (rel_error < 0.5 ? "FAIR" : "FAIL")
                    
                    @printf("%-10s | %8.0f | %9.0f | %8.0f | %7.3f | %s\n",
                            gene, initial_val, pred_val, target_val, rel_error, status)
                end
            end
            
            println("-"^70)
            
            # VitD summary
            println("\nVitamin D Dynamics:")
            @printf("  Initial:  %.3f (normalized)\n", vitd_initial)
            @printf("  Final:    %.3f (normalized)\n", vitd_final)
            @printf("  Retained: %.1f%%\n", vitd_retention)
            
            # Overall error
            if length(errors) > 0
                avg_error = mean(errors)
                println("\nOverall Performance:")
                @printf("  Average Error: %.4f\n", avg_error)
                
                if avg_error < 0.3
                    println("  Status:  EXCELLENT")
                elseif avg_error < 0.5
                    println("  Status:  GOOD")
                elseif avg_error < 1.0
                    println("  Status:  FAIR")
                else
                    println("  Status:  POOR")
                end
            else
                avg_error = NaN
            end
            
            return (
                error=avg_error,
                predicted=predicted,
                targets=targets,
                target_stds=target_stds,
                target_vals=target_vals,
                initial=u0_test[1:11],
                initial_vals=initial_vals,  # ← ADDED for boxplots
                vitd_initial=vitd_initial,
                vitd_final=vitd_final,
                sol=sol,
                valid_genes=valid_genes,
                errors=errors,
                gene_trajectories=gene_trajectories,
                vitd_trajectory=vitd_trajectory,
                time_points=sol.t,
                start_tp=start_tp,  # ← ADDED
                end_tp=end_tp       # ← ADDED
            )
            
        catch e
            println("\n Simulation Error:")
            println(e)
            return nothing
        end
    end

    # ============================================================================
    # RUN ALL TESTS
    # ============================================================================
    
    println("\n" * "="^80)
    println("RUNNING FIREPROOF VALIDATION TESTS")
    println("="^80)
    
    result1 = run_fireproof_test("d1", "d28", "R1", 27)
    result2 = run_fireproof_test("d29", "d56", "R2", 27)
    result3 = run_fireproof_test("d57", "d84", "R3", 27)

    # ============================================================================
    # VALIDATION SUMMARY
    # ============================================================================
    
    results = [result1, result2, result3]
    test_names = ["d1→d28 (R1)", "d29→d56 (R2)", "d57→d84 (R3)"]
    
    println("\n" * "="^80)
    println("FIREPROOF VALIDATION SUMMARY")
    println("="^80)
    
    valid_results = filter(r -> r !== nothing, results)
    
    if length(valid_results) > 0
        for (i, (name, result)) in enumerate(zip(test_names, results))
            if result !== nothing
                status = result.error < 0.5 ? " PASS" : " FAIL"
                @printf("  %s: %.4f %s\n", name, result.error, status)
            else
                println("  $name:  FAILED TO RUN")
            end
        end
        
        errors = [r.error for r in valid_results if !isnan(r.error)]
        if length(errors) > 0
            overall = mean(errors)
            println("-"^80)
            @printf("Overall Average Error: %.4f\n", overall)
            
            if overall < 0.3
                println(" EXCELLENT - Model captures inter-bolus dynamics!")
            elseif overall < 0.5
                println(" GOOD - Reasonable inter-bolus prediction")
            elseif overall < 1.0
                println(" FAIR - Some predictive capability")
            else
                println(" POOR - Model needs improvement")
            end
        end
    else
        println(" All validation tests failed")
    end

    # ============================================================================
    # CREATE ALL VALIDATION PLOTS
    # ============================================================================
    
    println("\n" * "="^80)
    println("CREATING FIREPROOF VALIDATION PLOTS")
    println("="^80)

    # ========================================================================
    # PLOT 1: BOXPLOT - PROPER FORMAT (Day X vs Day Y for each test)
    # ========================================================================
    
    if length(valid_results) >= 1
        println("\n Creating Validation Boxplots (CORRECTED FORMAT)...")
        
        # Process each test separately
        for (test_idx, result) in enumerate(results)
            if result !== nothing
                test_name = test_names[test_idx]
                rep = ["R1", "R2", "R3"][test_idx]
                
                # Color scheme for this test
                box_colors = [:lightblue, :lightcoral]
                marker_colors = [:darkblue, :darkred]
                
                plots_boxplot = []
                
                for (i, gene) in enumerate(gene_names)
                    p = plot(xlims=(0.5, 2.5),
                            xticks=([1, 2], [result.start_tp, result.end_tp]),
                            title=gene, titlefontsize=11,
                            xlabel=(i > 8 ? "Timepoint" : ""),
                            ylabel=(i in [1,5,9] ? "Expression (CPM)" : ""),
                            legend=false, grid=false, framestyle=:box,
                            guidefontsize=10, tickfontsize=9)
                    
                    # Initial values (start_tp)
                    initial_exp = result.initial_vals[gene]
                    initial_mean = mean(initial_exp)
                    
                    # Target values (end_tp)
                    target_exp = result.target_vals[gene]
                    target_mean = result.targets[i]
                    
                    # Simulated values
                    sim_initial = result.initial[i]
                    sim_final = result.predicted[i]
                    
                    # Position 1: Initial (start_tp)
                    if length(initial_exp) > 0
                        boxplot!(p, [1.0], [initial_exp],
                                color=box_colors[1], alpha=0.7,
                                bar_width=0.25, linewidth=1, outliers=true)
                        
                        # Mean line
                        plot!(p, [0.88, 1.12],
                              [initial_mean, initial_mean],
                              color=marker_colors[1], linewidth=4)
                    end
                    
                    # Simulated initial (diamond)
                    scatter!(p, [1.0], [sim_initial],
                            markersize=7, color=marker_colors[1],
                            marker=:diamond, markerstrokewidth=2, 
                            markerstrokecolor=:white)
                    
                    # Position 2: Final (end_tp)
                    if !isnan(target_mean) && length(target_exp) > 0
                        boxplot!(p, [2.0], [target_exp],
                                color=box_colors[2], alpha=0.7,
                                bar_width=0.25, linewidth=1, outliers=true)
                        
                        # Mean line
                        plot!(p, [1.88, 2.12],
                              [target_mean, target_mean],
                              color=marker_colors[2], linewidth=4)
                    end
                    
                    # Simulated final (diamond)
                    scatter!(p, [2.0], [sim_final],
                            markersize=7, color=marker_colors[2],
                            marker=:diamond, markerstrokewidth=2, 
                            markerstrokecolor=:white)
                    
                    push!(plots_boxplot, p)
                end
                
                # Add empty plot for 12th position
                push!(plots_boxplot, plot(framestyle=:none, grid=false, showaxis=false))
                
                final_boxplot = plot(plots_boxplot..., layout=grid(3,4), 
                                     size=(1600, 900), dpi=300)
                plot!(final_boxplot, 
                      plot_title="Fireproof Validation: $test_name (27-day prediction)")
                
                savefig(final_boxplot, "Fireproof_Validation/validation_boxplot_$(rep).png")
                savefig(final_boxplot, "Fireproof_Validation/validation_boxplot_$(rep).svg")
                println("   Saved validation_boxplot_$(rep).png and .svg")
            end
        end
        
        # Create combined legend
        legend_plot = plot(framestyle=:none, grid=false, showaxis=false)
        scatter!(legend_plot, [], [], color=:lightblue, marker=:square, 
                markersize=10, markerstrokewidth=0, label="Experimental (boxplot)")
        plot!(legend_plot, [], [], color=:darkblue, linewidth=4, label="Experimental mean")
        scatter!(legend_plot, [], [], color=:darkblue, marker=:diamond, 
                markersize=8, markerstrokewidth=2, markerstrokecolor=:white,
                label="Simulated")
        plot!(legend_plot, legend=:inside, legendcolumns=1, legendfontsize=12)
        savefig(legend_plot, "Fireproof_Validation1/validation_legend.png")
        println("   Saved validation_legend.png")
    end
    
    # ========================================================================
    # PLOT 2: ERROR BAR CHART
    # ========================================================================
    
    if length(valid_results) > 0
        println("\n Creating Error Summary Plot...")
        
        errors_by_test = []
        for result in results
            if result !== nothing
                push!(errors_by_test, result.error)
            else
                push!(errors_by_test, NaN)
            end
        end
        
        valid_idx = findall(!isnan, errors_by_test)
        
        if length(valid_idx) > 0
            p_summary = plot(size=(800, 500), dpi=300,
                            title="Fireproof Validation: Average Errors",
                            xlabel="Test", ylabel="Average Relative Error",
                            titlefontsize=14, guidefontsize=12,
                            legend=:topright)
            
            colors = [e < 0.3 ? :green : (e < 0.5 ? :orange : :red) 
                     for e in errors_by_test[valid_idx]]
            
            bar!(p_summary, valid_idx, errors_by_test[valid_idx],
                 color=colors, alpha=0.7, width=0.6, label="")
            
            hline!(p_summary, [0.3], line=:dash, color=:green, 
                   linewidth=2, label="Excellent (<0.3)")
            hline!(p_summary, [0.5], line=:dash, color=:orange, 
                   linewidth=2, label="Good (<0.5)")
            hline!(p_summary, [1.0], line=:dash, color=:red, 
                   linewidth=2, label="Fair (<1.0)")
            
            plot!(p_summary, xticks=(1:3, ["d1→d28\n(R1)", "d29→d56\n(R2)", "d57→d84\n(R3)"]))
            
            # Annotate with values
            for (i, idx) in enumerate(valid_idx)
                annotate!(p_summary, idx, errors_by_test[idx] + 0.05,
                         text(@sprintf("%.3f", errors_by_test[idx]), 10, :center))
            end
            
            savefig(p_summary, "Fireproof_Validation1/error_summary.png")
            savefig(p_summary, "Fireproof_Validation1/error_summary.svg")
            println("  Saved error_summary.png and .svg")
        end
    end
    
    # ========================================================================
    # PLOT 3: PER-GENE ERROR HEATMAP
    # ========================================================================
    
    if length(valid_results) >= 1
        println("\n Creating Per-Gene Error Heatmap...")
        
        # Create error matrix
        error_matrix = zeros(11, 3)
        for (test_idx, result) in enumerate(results)
            if result !== nothing
                for (gene_idx, gene) in enumerate(gene_names)
                    if gene in result.valid_genes
                        idx = findfirst(==(gene), result.valid_genes)
                        error_matrix[gene_idx, test_idx] = result.errors[idx]
                    else
                        error_matrix[gene_idx, test_idx] = NaN
                    end
                end
            else
                error_matrix[:, test_idx] .= NaN
            end
        end
        
        # Create heatmap
        test_labels_hm = ["d1→d28", "d29→d56", "d57→d84"]
        p_heatmap = heatmap(error_matrix',
                           c=:RdYlGn_11,
                           clims=(0, 1.0),
                           xlabel="Gene",
                           ylabel="Test",
                           title="Per-Gene Validation Errors",
                           xticks=(1:11, gene_names),
                           yticks=(1:3, test_labels_hm),
                           xrotation=45,
                           size=(1000, 400),
                           dpi=300,
                           colorbar_title="Relative Error")
        
        savefig(p_heatmap, "Fireproof_Validation1/error_heatmap.png")
        savefig(p_heatmap, "Fireproof_Validation1/error_heatmap.svg")
        println("  Saved error_heatmap.png and .svg")
    end
    
    # ========================================================================
    # PLOT 4: GENE TRAJECTORIES OVER 27 DAYS
    # ========================================================================
    
    if length(valid_results) >= 1
        println("\n Creating Gene Trajectory Plots...")
        
        for (test_idx, result) in enumerate(results)
            if result !== nothing
                test_name = test_names[test_idx]
                rep = ["R1", "R2", "R3"][test_idx]
                
                # Create 11-gene trajectory plot
                plots_traj = []
                
                for (i, gene) in enumerate(gene_names)
                    p = plot(size=(400, 300),
                            title=gene, titlefontsize=10,
                            xlabel=(i > 8 ? "Days" : ""),
                            ylabel=(i in [1,5,9] ? "Expression" : ""),
                            legend=false, grid=true, framestyle=:box,
                            guidefontsize=9, tickfontsize=8)
                    
                    traj = result.gene_trajectories[gene]
                    t = result.time_points
                    
                    # Plot trajectory
                    plot!(p, t, traj, color=:blue, linewidth=2)
                    
                    # Mark start and end
                    scatter!(p, [t[1]], [traj[1]], color=:green, 
                            markersize=5, marker=:circle)
                    scatter!(p, [t[end]], [traj[end]], color=:red, 
                            markersize=5, marker=:square)
                    
                    # Add target line if available
                    if !isnan(result.targets[i])
                        hline!(p, [result.targets[i]], line=:dash, 
                              color=:orange, linewidth=1.5)
                    end
                    
                    push!(plots_traj, p)
                end
                
                # Add empty plot
                push!(plots_traj, plot(framestyle=:none, grid=false, showaxis=false))
                
                final_traj = plot(plots_traj..., layout=grid(3,4), 
                                 size=(1600, 900), dpi=300)
                plot!(final_traj, plot_title="Gene Trajectories: $test_name")
                
                savefig(final_traj, "Fireproof_Validation1/trajectories_$(rep).png")
                savefig(final_traj, "Fireproof_Validation1/trajectories_$(rep).svg")
                println("  Saved trajectories_$(rep).png and .svg")
            end
        end
    end
    
    # ========================================================================
    # PLOT 5: VITD DECAY COMPARISON
    # ========================================================================
    
    if length(valid_results) >= 1
        println("\nCreating VitD Decay Comparison...")
        
        p_vitd = plot(size=(900, 600), dpi=300,
                     title="Vitamin D Decay: All Tests",
                     xlabel="Days", ylabel="VitD Level (normalized)",
                     titlefontsize=14, guidefontsize=12,
                     legend=:topright, grid=true)
        
        test_colors = [:blue, :green, :red]
        
        for (test_idx, result) in enumerate(results)
            if result !== nothing
                plot!(p_vitd, result.time_points, result.vitd_trajectory,
                     color=test_colors[test_idx], linewidth=2,
                     label=test_names[test_idx])
                
                # Add half-life annotation
                rep = ["R1", "R2", "R3"][test_idx]
                half_life = log(2) / vitd_validation[rep]
                
                if test_idx <= 3
                    annotate!(p_vitd, 20, 0.9 - (test_idx-1)*0.1,
                             text("$(test_names[test_idx]): t½=$(round(half_life, digits=1))d", 
                                  10, test_colors[test_idx]))
                end
            end
        end
        
        savefig(p_vitd, "Fireproof_Validation1/vitd_decay_comparison.png")
        savefig(p_vitd, "Fireproof_Validation1/vitd_decay_comparison.svg")
        println("  Saved vitd_decay_comparison.png and .svg")
    end
    
    # ========================================================================
    # PLOT 6: FOLD CHANGE COMPARISON
    # ========================================================================
    
    if length(valid_results) >= 1
        println("\n Creating Fold Change Comparison...")
        
        for (test_idx, result) in enumerate(results)
            if result !== nothing
                test_name = test_names[test_idx]
                rep = ["R1", "R2", "R3"][test_idx]
                
                p_fold = plot(size=(900, 600), dpi=300,
                             title="Fold Changes: $test_name",
                             xlabel="Gene", ylabel="Fold Change",
                             titlefontsize=14, guidefontsize=12,
                             bottom_margin=10Plots.mm,
                             legend=:topright)
                
                x_pos = 1:11
                pred_fold = result.predicted ./ (result.initial .+ 1e-6)
                target_fold = result.targets ./ (result.initial .+ 1e-6)
                
                # Bar plot for predicted
                bar!(p_fold, x_pos .- 0.2, pred_fold,
                     width=0.4, label="Predicted", color=:blue, alpha=0.7)
                
                # Bar plot for target
                bar!(p_fold, x_pos .+ 0.2, target_fold,
                     width=0.4, label="Target", color=:orange, alpha=0.7)
                
                # No change line
                hline!(p_fold, [1.0], line=:dash, color=:gray, 
                      linewidth=2, label="No change")
                
                plot!(p_fold, xticks=(x_pos, gene_names), xrotation=45)
                
                savefig(p_fold, "Fireproof_Validation1/fold_changes_$(rep).png")
                savefig(p_fold, "Fireproof_Validation1/fold_changes_$(rep).svg")
                println("  Saved fold_changes_$(rep).png and .svg")
            end
        end
    end
    
    # ========================================================================
    # PLOT 7: COMPARISON BARS (Initial vs Predicted vs Target)
    # ========================================================================
    
    if length(valid_results) >= 1
        println("\n Creating Comparison Bar Plots...")
        
        for (test_idx, result) in enumerate(results)
            if result !== nothing
                test_name = test_names[test_idx]
                rep = ["R1", "R2", "R3"][test_idx]
                
                p_comp = plot(size=(1000, 600), dpi=300,
                             title="Expression Comparison: $test_name",
                             xlabel="Gene", ylabel="Expression (CPM)",
                             titlefontsize=14, guidefontsize=12,
                             bottom_margin=10Plots.mm,
                             legend=:topright)
                
                x_pos = 1:11
                
                # Three bars per gene
                bar!(p_comp, x_pos .- 0.3, result.initial,
                     width=0.3, label="Initial ($(result.start_tp))", 
                     color=:lightblue, alpha=0.8)
                
                bar!(p_comp, x_pos, result.predicted,
                     width=0.3, label="Predicted ($(result.end_tp))", 
                     color=:orange, alpha=0.8)
                
                bar!(p_comp, x_pos .+ 0.3, result.targets,
                     width=0.3, label="Target ($(result.end_tp))", 
                     color=:green, alpha=0.8)
                
                plot!(p_comp, xticks=(x_pos, gene_names), xrotation=45)
                
                savefig(p_comp, "Fireproof_Validation1/comparison_bars_$(rep).png")
                savefig(p_comp, "Fireproof_Validation1/comparison_bars_$(rep).svg")
                println("  Saved comparison_bars_$(rep).png and .svg")
            end
        end
    end
    
    println("\nAll fireproof validation plots complete!")
    println("   Results saved in Fireproof_Validation/")

end # End of validation file check

println("\n" * "="^80)
println("FIREPROOF VALIDATION COMPLETE")
println("="^80)

In [ ]:
# ================================================================================
# MODIFIED PLOTTING CELLS
#
# FONT STRATEGY (definitive):
#   GR has a fixed set of built-in fonts — no TTF file lookup.
#   Full list: https://gr-framework.org/fonts.html
#   For bold sans-serif: fontfamily = "helvetica bold"  (GR index 107)
#   This is passed via Plots as: font(size, "helvetica bold")
#   This NEVER causes a "could not find font" error.
#
# AXES: framestyle = :axes  → bottom + left lines only (L-shape)
#
# GAP LAYOUT (4 units between t0/t24, large gaps between boluses):
#   R1:  d0=0,  d1=4
#   R2:  d28=15, d29=19
#   R3:  d56=30, d57=34
#
# All outputs saved as .png + .svg
# ================================================================================

# ============================================================================
# BLOCK 0 — HELPERS
# ============================================================================

function _get_std(gene_row, raw_data, prefix)
    nrow(gene_row) == 0 && return 0.0
    row  = gene_row[1, :]
    cols = filter(c -> startswith(string(c), prefix), names(raw_data))
    vals = [convert_european_decimal(row[c]) for c in cols if !ismissing(row[c])]
    vals = filter(v -> !isnan(v) && v > 0, vals)
    return length(vals) > 1 ? std(vals) : 0.0
end

# GR built-in bold font — confirmed at gr-framework.org/fonts.html
const BF = "helvetica bold"   # index 107, always available, no TTF needed

function apply_plot_style!(p;
        title     = "",
        xlabel    = "",
        ylabel    = "",
        titlefs   = 16,
        guidefs   = 15,
        tickfs    = 13,
        xticks    = nothing,
        yticks    = nothing,
        xlims     = nothing,
        ylims     = nothing,
        xrotation = 0)

    plot!(p;
        framestyle            = :axes,
        grid                  = false,
        foreground_color_axis   = :black,
        foreground_color_border = :black,
        foreground_color_text   = :black,
        title         = title,
        titlefont     = font(titlefs, BF),
        xlabel        = xlabel,
        ylabel        = ylabel,
        guidefont     = font(guidefs, BF),
        tickfont      = font(tickfs,  BF),
        legendfont    = font(13,      BF),
        xrotation     = xrotation,
        left_margin   = 14Plots.mm,
        bottom_margin = 14Plots.mm,
    )
    xlims  !== nothing && xlims!(p,  xlims)
    ylims  !== nothing && ylims!(p,  ylims)
    xticks !== nothing && xticks!(p, xticks)
    yticks !== nothing && yticks!(p, yticks)
    return p
end

function add_spines!(p; lw=6)
    return p
end

function save_both(plt, base_path)
    savefig(plt, base_path * ".png")
    savefig(plt, base_path * ".svg")
    println("  $(base_path).png + .svg")
end

println("Helpers loaded  (font: \"$BF\")")


# ============================================================================
# BLOCK 1 — SETUP
# ============================================================================

using Plots, StatsPlots
gr()

for d in ["Gene_model_plots", "Fireproof_Validation"]
    isdir(d) || mkdir(d)
end

sim_results       = get_simulation_results()
exp_distributions = get_experimental_distributions()

R1_x = [0,  4]
R2_x = [15, 19]
R3_x = [30, 34]

GAP_XTICKS = ([0, 4, 15, 19, 30, 34],
              ["d0","d1","d28","d29","d56","d57"])
GAP_XLIMS  = (-1, 36)
# ---------------------------------------------------------------------------
# COLOUR PALETTE  — all built from the original vspan base RGB values:
#   R1 base: (0.39, 0.58, 0.93)  — cornflower blue
#   R2 base: (0.68, 0.87, 0.68)  — fresh medium green
#   R3 base: (0.98, 0.85, 0.37)  — warm golden yellow
#
# Boxplot pairs  (light = t0/Day1 ;  dark = t24/Day28)
#   light  = base colour at α 0.45  (airy, tinted)
#   dark   = base colour fully saturated, darkened ~40%
# ---------------------------------------------------------------------------

# Background spans — gap layout (dynamics)  ← UNCHANGED from original
GAP_VSPAN  = [
    (0,  4,  RGBA(0.39, 0.58, 0.93, 0.12)),
    (15, 19, RGBA(0.68, 0.87, 0.68, 0.12)),
    (30, 34, RGBA(0.98, 0.85, 0.37, 0.12)),
]

TRAJ_XTICKS = ([0,14,28,42,56,70,84],
               ["0","14","28","42","56","70","84"])

# Background spans — 84-day trajectory  ← UNCHANGED from original
TRAJ_VSPAN  = [
    (0,  1,  RGBA(0.39, 0.58, 0.93, 0.15)),
    (1,  28, RGBA(0.39, 0.58, 0.93, 0.08)),
    (28, 29, RGBA(0.68, 0.87, 0.68, 0.15)),
    (29, 56, RGBA(0.68, 0.87, 0.68, 0.08)),
    (56, 57, RGBA(0.98, 0.85, 0.37, 0.15)),
    (57, 84, RGBA(0.98, 0.85, 0.37, 0.08)),
]

# Scatter/line colours for timeline & dynamics — full-opacity base colours
rep_colors  = Dict(
    "R1" => RGBA(0.39, 0.58, 0.93, 1.0),   # cornflower blue
    "R2" => RGBA(0.68, 0.87, 0.68, 1.0),   # fresh green (with dark stroke for visibility)
    "R3" => RGBA(0.98, 0.85, 0.37, 1.0),   # golden yellow (with dark stroke for visibility)
)
rep_markers = Dict("R1"=>:circle, "R2"=>:square, "R3"=>:diamond)

# Boxplot fill colours — [t0/Day1 (light),  t24/Day28 (dark)]
#   light = base RGB mixed with white  (α 0.50 on white background ≈ pastel)
#   dark  = base RGB scaled down ~45%  (rich, saturated)
box_colors = Dict(
    "R1" => [RGBA(0.39, 0.58, 0.93, 0.45),   # light cornflower blue
             RGBA(0.18, 0.30, 0.62, 0.85)],  # deep royal blue
    "R2" => [RGBA(0.68, 0.87, 0.68, 0.50),   # light mint green
             RGBA(0.25, 0.58, 0.25, 0.85)],  # deep forest green
    "R3" => [RGBA(0.98, 0.85, 0.37, 0.50),   # light golden yellow
             RGBA(0.65, 0.48, 0.04, 0.85)],  # dark mustard / amber
)

# Mean-line & simulated-diamond colours (opaque, darker than box fill)
marker_colors = Dict(
    "R1" => [RGBA(0.39, 0.58, 0.93, 1.0),   # cornflower blue  (t0)
             RGBA(0.18, 0.30, 0.62, 1.0)],  # deep royal blue  (t24)
    "R2" => [RGBA(0.25, 0.65, 0.25, 1.0),   # medium green     (t0)
             RGBA(0.15, 0.42, 0.15, 1.0)],  # deep forest green (t24)
    "R3" => [RGBA(0.85, 0.68, 0.10, 1.0),   # golden           (t0)
             RGBA(0.55, 0.40, 0.02, 1.0)],  # dark amber       (t24)
)

println("Setup complete")


# ============================================================================
# BLOCK 2 — TIMELINE PLOT
# ============================================================================

println("\n Timeline Plot...")

plots_timeline = []

for (i, gene) in enumerate(gene_names)

    p = plot(; legend=false)
    apply_plot_style!(p;
        title=gene, titlefs=16,
        xlabel=(i>8 ? "Time" : ""),
        ylabel=(i in [1,5,9] ? "Expression (CPM)" : ""),
        guidefs=15, tickfs=13,
        xticks=GAP_XTICKS, xlims=GAP_XLIMS, xrotation=45)

    for (x0,x1,col) in GAP_VSPAN
        vspan!(p,[x0,x1]; fillcolor=col, linewidth=0, label="")
    end

    rep_xpos = Dict("R1"=>R1_x, "R2"=>R2_x, "R3"=>R3_x)

    for rep in ["R1","R2","R3"]
        haskey(sim_results, rep) || continue
        sim_t0  = sim_results[rep][i,1]
        sim_t24 = sim_results[rep][i,2]
        exp_t0  = exp_distributions[rep][gene]["t0"]
        exp_t24 = exp_distributions[rep][gene]["t24"]
        m0, m24   = mean(exp_t0), mean(exp_t24)
        sd0, sd24 = (length(exp_t0)>1 ? std(exp_t0) : 0.0),
                    (length(exp_t24)>1 ? std(exp_t24) : 0.0)
        x0v, x24v = rep_xpos[rep]

        scatter!(p,[x0v,x24v],[m0,m24];
                yerror=[sd0,sd24], markersize=7,
                color=rep_colors[rep], marker=rep_markers[rep],
                markerstrokewidth=0, alpha=0.85, label="")
        scatter!(p,[x0v,x24v],[sim_t0,sim_t24];
                markersize=9, color=rep_colors[rep],
                marker=:star5, markerstrokewidth=1,
                markerstrokecolor=:black, label="")
        plot!(p,[x0v,x24v],[sim_t0,sim_t24];
              color=rep_colors[rep], linestyle=:dash,
              linewidth=1.5, alpha=0.6, label="")
    end
    add_spines!(p)
    push!(plots_timeline, p)
end

push!(plots_timeline,
      plot(framestyle=:none, grid=false, showaxis=false,
           background_color=:transparent))

final_tl = plot(plots_timeline...;
                layout=grid(3,4), size=(1800,1000), dpi=300,
                plot_title="Timeline Summary: 24-Hour Expression Changes",
                plot_titlefont=font(16, BF))
save_both(final_tl, "Gene_model_plots/Timeline_Summary")

leg_tl = plot(framestyle=:none, grid=false, showaxis=false,
              size=(820,140), dpi=300, background_color=:transparent)
for rep in ["R1","R2","R3"]
    scatter!(leg_tl,[],[]; color=rep_colors[rep], marker=rep_markers[rep],
            markersize=7, markerstrokewidth=0,
            label="$rep Experimental (mean\u00b1SD)")
end
scatter!(leg_tl,[],[]; color=:gray, marker=:star5, markersize=9,
        markerstrokewidth=1, markerstrokecolor=:black, label="Simulated")
plot!(leg_tl; legend=:inside, legendcolumns=4, legendfont=font(13, BF))
save_both(leg_tl, "Gene_model_plots/Timeline_Legend")


# ============================================================================
# BLOCK 3 — COMBINED BOXPLOT
# ============================================================================

println("\n📊 Combined Boxplot...")

plots_box = []

for (i, gene) in enumerate(gene_names)

    p = plot(; legend=false)
    apply_plot_style!(p;
        title=gene, titlefs=16,
        xlabel=(i>8 ? "Time" : ""),
        ylabel=(i in [1,5,9] ? "Expression (CPM)" : ""),
        guidefs=15, tickfs=13,
        xticks=([1,2],["0 h","24 h"]), xlims=(0.5,2.5))

    pos = Dict("R1"=>[0.8,1.8], "R2"=>[1.0,2.0], "R3"=>[1.2,2.2])

    for rep in ["R1","R2","R3"]
        haskey(sim_results, rep) || continue
        sim_t0  = sim_results[rep][i,1]
        sim_t24 = sim_results[rep][i,2]
        exp_t0  = exp_distributions[rep][gene]["t0"]
        exp_t24 = exp_distributions[rep][gene]["t24"]
        m0, m24 = mean(exp_t0), mean(exp_t24)

        boxplot!(p,[pos[rep][1]],[exp_t0]; color=box_colors[rep][1],
                alpha=0.7, bar_width=0.15, linewidth=1, outliers=true, label="")
        boxplot!(p,[pos[rep][2]],[exp_t24]; color=box_colors[rep][2],
                alpha=0.7, bar_width=0.15, linewidth=1, outliers=true, label="")
        plot!(p,[pos[rep][1]-0.08,pos[rep][1]+0.08],[m0,m0];
              color=marker_colors[rep][1], linewidth=4, label="")
        plot!(p,[pos[rep][2]-0.08,pos[rep][2]+0.08],[m24,m24];
              color=marker_colors[rep][2], linewidth=4, label="")
        scatter!(p,[pos[rep][1]],[sim_t0];
                markersize=7, color=marker_colors[rep][1], marker=:diamond,
                markerstrokewidth=1.5, markerstrokecolor=:white, label="")
        scatter!(p,[pos[rep][2]],[sim_t24];
                markersize=7, color=marker_colors[rep][2], marker=:diamond,
                markerstrokewidth=1.5, markerstrokecolor=:white, label="")
    end
    add_spines!(p)
    push!(plots_box, p)
end

push!(plots_box,
      plot(framestyle=:none, grid=false, showaxis=false,
           background_color=:transparent))

final_box = plot(plots_box...;
                 layout=grid(3,4), size=(1800,1000), dpi=300,
                 plot_title="Combined Boxplot: All Replicates",
                 plot_titlefont=font(16, BF))
save_both(final_box, "Gene_model_plots/Combined_Boxplot")

leg_box = plot(framestyle=:none, grid=false, showaxis=false,
               size=(1000,150), dpi=300, background_color=:transparent)
for rep in ["R1","R2","R3"]
    scatter!(leg_box,[],[]; color=box_colors[rep][1], marker=:square,
            markersize=10, markerstrokewidth=0, label="$rep Exp 0 h")
    scatter!(leg_box,[],[]; color=box_colors[rep][2], marker=:square,
            markersize=10, markerstrokewidth=0, label="$rep Exp 24 h")
end
scatter!(leg_box,[],[]; color=:gray, marker=:diamond, markersize=8,
        markerstrokewidth=1.5, markerstrokecolor=:white, label="Simulated")
plot!(leg_box; legend=:inside, legendcolumns=4, legendfont=font(13, BF))
save_both(leg_box, "Gene_model_plots/Boxplot_Legend")


# ============================================================================
# BLOCK 4 — FIREPROOF VALIDATION BOXPLOT
# ============================================================================

println("\n📊 Validation Boxplot...")

if length(valid_results) >= 1

    plots_vbox = []

    for (i, gene) in enumerate(gene_names)

        p = plot(; legend=false)
        apply_plot_style!(p;
            title=gene, titlefs=16,
            xlabel=(i>8 ? "Time" : ""),
            ylabel=(i in [1,5,9] ? "Expression (CPM)" : ""),
            guidefs=15, tickfs=13,
            xticks=([1,2],["Day 1","Day 28"]), xlims=(0.5,2.5))

        pos = Dict("R1"=>[0.8,1.8], "R2"=>[1.0,2.0], "R3"=>[1.2,2.2])

        for (rep, result) in zip(["R1","R2","R3"],[result1,result2,result3])
            result === nothing && continue
            iv = result.initial_vals[gene]
            tv = result.target_vals[gene]
            tm = result.targets[i]
            si = result.initial[i]
            sf = result.predicted[i]

            if length(iv) > 0
                boxplot!(p,[pos[rep][1]],[iv]; color=box_colors[rep][1],
                        alpha=0.7, bar_width=0.15, linewidth=1, outliers=true, label="")
                plot!(p,[pos[rep][1]-0.08,pos[rep][1]+0.08],[mean(iv),mean(iv)];
                      color=marker_colors[rep][1], linewidth=4, label="")
            end
            scatter!(p,[pos[rep][1]],[si];
                    markersize=7, color=marker_colors[rep][1], marker=:diamond,
                    markerstrokewidth=1.5, markerstrokecolor=:white, label="")

            if !isnan(tm) && length(tv) > 0
                boxplot!(p,[pos[rep][2]],[tv]; color=box_colors[rep][2],
                        alpha=0.7, bar_width=0.15, linewidth=1, outliers=true, label="")
                plot!(p,[pos[rep][2]-0.08,pos[rep][2]+0.08],[tm,tm];
                      color=marker_colors[rep][2], linewidth=4, label="")
            end
            scatter!(p,[pos[rep][2]],[sf];
                    markersize=7, color=marker_colors[rep][2], marker=:diamond,
                    markerstrokewidth=1.5, markerstrokecolor=:white, label="")
        end
        add_spines!(p)
    push!(plots_vbox, p)
    end

    push!(plots_vbox,
          plot(framestyle=:none, grid=false, showaxis=false,
               background_color=:transparent))

    final_vbox = plot(plots_vbox...;
                      layout=grid(3,4), size=(1800,1000), dpi=300,
                      plot_title="Fireproof Validation: Inter-bolus Predictions",
                      plot_titlefont=font(16, BF))
    save_both(final_vbox, "Fireproof_Validation/validation_boxplot_combined")

    leg_vbox = plot(framestyle=:none, grid=false, showaxis=false,
                    size=(1000,150), dpi=300, background_color=:transparent)
    for rep in ["R1","R2","R3"]
        scatter!(leg_vbox,[],[]; color=box_colors[rep][1], marker=:square,
                markersize=10, markerstrokewidth=0, label="$rep Exp (start)")
        scatter!(leg_vbox,[],[]; color=box_colors[rep][2], marker=:square,
                markersize=10, markerstrokewidth=0, label="$rep Exp (end)")
    end
    scatter!(leg_vbox,[],[]; color=:gray, marker=:diamond, markersize=8,
            markerstrokewidth=1.5, markerstrokecolor=:white, label="Simulated")
    plot!(leg_vbox; legend=:inside, legendcolumns=4, legendfont=font(13, BF))
    save_both(leg_vbox, "Fireproof_Validation/validation_legend")
end


# ============================================================================
# BLOCK 5 — 84-DAY TRAJECTORY
# ============================================================================

println("\n📊 84-Day Trajectory...")

if length(valid_results) >= 3

    opt_sol_R1 = solve_model("R1", optimized_params["R1"])
    opt_sol_R2 = solve_model("R2", optimized_params["R2"])
    opt_sol_R3 = solve_model("R3", optimized_params["R3"])

    if opt_sol_R1 !== nothing && opt_sol_R2 !== nothing && opt_sol_R3 !== nothing

        function _save_individual_trajectory(gene,
                                             exp_days, exp_means, exp_stds,
                                             pred_days, pred_vals)
            pi = plot(; legend=:topright, size=(900,600), dpi=300,
                       bottom_margin=16Plots.mm, left_margin=16Plots.mm)
            apply_plot_style!(pi;
                title=gene, titlefs=18, xlabel="Days",
                ylabel="Expression (CPM)", guidefs=16, tickfs=14,
                xticks=TRAJ_XTICKS, xlims=(-2,86), xrotation=45)
            for (x0,x1,col) in TRAJ_VSPAN
                vspan!(pi,[x0,x1]; fillcolor=col, linewidth=0, label="")
            end
            for (d,m,s) in zip(exp_days,exp_means,exp_stds)
                s>0 && !isnan(s) &&
                    plot!(pi,[d,d],[m-s,m+s];
                          color=RGBA(1.0,0.0,0.0,0.3), linewidth=2, label="")
            end
            scatter!(pi,exp_days,exp_means; markersize=10, markershape=:square,
                    markercolor=:red, markerstrokewidth=0, alpha=0.8,
                    label="Experimental (mean\u00b1SD)")
            plot!(pi,pred_days,pred_vals; color=:black, linewidth=3, label="")
            scatter!(pi,pred_days,pred_vals; markersize=9, markershape=:circle,
                    markercolor=:black, markerstrokewidth=0, label="Predicted")
            add_spines!(pi)
            save_both(pi, "Fireproof_Validation/trajectory_$(gene)")
        end

        plots_traj = []

        for (i, gene) in enumerate(gene_names)
            gene_row  = filter(r -> r.hgnc_symbol == gene, raw_data)
            exp_days  = [0,1,28,29,56,57,84]
            exp_means = [experimental_data["R1"][i,1],
                         experimental_data["R1"][i,2],
                         result1.targets[i],
                         experimental_data["R2"][i,2],
                         result2.targets[i],
                         experimental_data["R3"][i,2],
                         result3.targets[i]]
            exp_stds  = [_get_std(gene_row,raw_data,"R1_0_"),
                         _get_std(gene_row,raw_data,"R1_24_"),
                         result1.target_stds[i],
                         _get_std(gene_row,raw_data,"R2_24_"),
                         result2.target_stds[i],
                         _get_std(gene_row,raw_data,"R3_24_"),
                         result3.target_stds[i]]
            pred_days = [0,1,28,29,56,57,84]
            pred_vals = [opt_sol_R1.u[1][i], opt_sol_R1.u[2][i],
                         result1.predicted[i], opt_sol_R2.u[2][i],
                         result2.predicted[i], opt_sol_R3.u[2][i],
                         result3.predicted[i]]

            p = plot(; legend=false)
            apply_plot_style!(p;
                title=gene, titlefs=16,
                xlabel=(i>8 ? "Days" : ""),
                ylabel=(i in [1,5,9] ? "Expression (CPM)" : ""),
                guidefs=15, tickfs=13,
                xticks=TRAJ_XTICKS, xlims=(-2,86), xrotation=45)
            for (x0,x1,col) in TRAJ_VSPAN
                vspan!(p,[x0,x1]; fillcolor=col, linewidth=0, label="")
            end
            for (d,m,s) in zip(exp_days,exp_means,exp_stds)
                s>0 && !isnan(s) &&
                    plot!(p,[d,d],[m-s,m+s];
                          color=RGBA(1.0,0.0,0.0,0.3), linewidth=2, label="")
            end
            scatter!(p,exp_days,exp_means; markersize=8, markershape=:square,
                    markercolor=:red, markerstrokewidth=0, alpha=0.8, label="")
            plot!(p,pred_days,pred_vals; color=:black, linewidth=2.5, label="")
            scatter!(p,pred_days,pred_vals; markersize=7, markershape=:circle,
                    markercolor=:black, markerstrokewidth=0, label="")

            add_spines!(p)
    push!(plots_traj, p)
            _save_individual_trajectory(gene, exp_days, exp_means, exp_stds,
                                        pred_days, pred_vals)
        end
        println("  Individual trajectory .png+.svg for all 11 genes")

        push!(plots_traj,
              plot(framestyle=:none, grid=false, showaxis=false,
                   background_color=:transparent))

        final_traj = plot(plots_traj...;
                          layout=grid(3,4), size=(2000,1200), dpi=300,
                          plot_title="84-Day Trajectory: Optimization + Validation",
                          plot_titlefont=font(16, BF))
        save_both(final_traj, "Fireproof_Validation/trajectory_84day_combined")

        leg_tr = plot(framestyle=:none, grid=false, showaxis=false,
                      size=(520,130), dpi=300, background_color=:transparent)
        scatter!(leg_tr,[],[]; markershape=:square, markersize=12,
                markercolor=:red, alpha=0.8, markerstrokewidth=0,
                label="Experimental (mean\u00b1SD)")
        plot!(leg_tr,[],[]; color=:black, linewidth=3, marker=:circle,
              markersize=10, markercolor=:black, markerstrokewidth=0,
              label="Predicted")
        plot!(leg_tr; legend=:top, legendcolumns=2, legendfont=font(13, BF))
        save_both(leg_tr, "Fireproof_Validation/trajectory_legend")
    end
end


# ============================================================================
# BLOCK 6 — 24-HOUR DYNAMICS
# ============================================================================

println("\n📊 24-Hour Dynamics...")

if length(valid_results) >= 3

    opt_sol_R1 = solve_model("R1", optimized_params["R1"])
    opt_sol_R2 = solve_model("R2", optimized_params["R2"])
    opt_sol_R3 = solve_model("R3", optimized_params["R3"])

    if opt_sol_R1 !== nothing && opt_sol_R2 !== nothing && opt_sol_R3 !== nothing

        plots_24h = []

        for (i, gene) in enumerate(gene_names)
            gene_row  = filter(r -> r.hgnc_symbol == gene, raw_data)
            exp_xpos  = [0,4,15,19,30,34]
            exp_means = [experimental_data["R1"][i,1],
                         experimental_data["R1"][i,2],
                         experimental_data["R2"][i,1],
                         experimental_data["R2"][i,2],
                         experimental_data["R3"][i,1],
                         experimental_data["R3"][i,2]]
            exp_stds  = [_get_std(gene_row,raw_data,"R1_0_"),
                         _get_std(gene_row,raw_data,"R1_24_"),
                         _get_std(gene_row,raw_data,"R2_0_"),
                         _get_std(gene_row,raw_data,"R2_24_"),
                         _get_std(gene_row,raw_data,"R3_0_"),
                         _get_std(gene_row,raw_data,"R3_24_")]

            # Panel for combined figure
            p = plot(; legend=false)
            apply_plot_style!(p;
                title=gene, titlefs=16,
                xlabel=(i>8 ? "Time" : ""),
                ylabel=(i in [1,5,9] ? "Expression (CPM)" : ""),
                guidefs=15, tickfs=13,
                xticks=GAP_XTICKS, xlims=GAP_XLIMS, xrotation=45)
            for (x0,x1,col) in GAP_VSPAN
                vspan!(p,[x0,x1]; fillcolor=col, linewidth=0, label="")
            end
            for (d,m,s) in zip(exp_xpos,exp_means,exp_stds)
                s>0 && !isnan(s) &&
                    plot!(p,[d,d],[m-s,m+s];
                          color=RGBA(1.0,0.0,0.0,0.3), linewidth=2, label="")
            end
            scatter!(p,exp_xpos,exp_means; markersize=8, markershape=:square,
                    markercolor=:red, markerstrokewidth=0, alpha=0.8, label="")
            for (xpair, sol) in zip(([0,4],[15,19],[30,34]),
                                     (opt_sol_R1,opt_sol_R2,opt_sol_R3))
                vals = [sol.u[1][i], sol.u[2][i]]
                plot!(p,xpair,vals; color=:black, linewidth=2.5, label="")
                scatter!(p,xpair,vals; markersize=7, markershape=:circle,
                        markercolor=:black, markerstrokewidth=0, label="")
            end
            add_spines!(p)
    push!(plots_24h, p)

            # Individual gene plot
            pi = plot(; legend=:topright, size=(900,600), dpi=300,
                       bottom_margin=16Plots.mm, left_margin=16Plots.mm)
            apply_plot_style!(pi;
                title=gene, titlefs=18, xlabel="Time",
                ylabel="Expression (CPM)", guidefs=16, tickfs=14,
                xticks=GAP_XTICKS, xlims=GAP_XLIMS, xrotation=45)
            for (x0,x1,col) in GAP_VSPAN
                vspan!(pi,[x0,x1]; fillcolor=col, linewidth=0, label="")
            end
            for (d,m,s) in zip(exp_xpos,exp_means,exp_stds)
                s>0 && !isnan(s) &&
                    plot!(pi,[d,d],[m-s,m+s];
                          color=RGBA(1.0,0.0,0.0,0.3), linewidth=2, label="")
            end
            scatter!(pi,exp_xpos,exp_means; markersize=10, markershape=:square,
                    markercolor=:red, markerstrokewidth=0, alpha=0.8,
                    label="Experimental (mean\u00b1SD)")
            for (xpair, sol) in zip(([0,4],[15,19],[30,34]),
                                     (opt_sol_R1,opt_sol_R2,opt_sol_R3))
                vals = [sol.u[1][i], sol.u[2][i]]
                plot!(pi,xpair,vals; color=:black, linewidth=3, label="")
                scatter!(pi,xpair,vals; markersize=9, markershape=:circle,
                        markercolor=:black, markerstrokewidth=0, label="Predicted")
            end
            add_spines!(pi)
            save_both(pi, "Fireproof_Validation/dynamics_24h_$(gene)")
        end
        println("  ✅ Individual 24h .png+.svg for all 11 genes")

        push!(plots_24h,
              plot(framestyle=:none, grid=false, showaxis=false,
                   background_color=:transparent))

        final_24h = plot(plots_24h...;
                         layout=grid(3,4), size=(2000,1200), dpi=300,
                         plot_title="24-Hour Dynamics: Three Bolus Periods",
                         plot_titlefont=font(16, BF))
        save_both(final_24h, "Fireproof_Validation/dynamics_24h_combined")

        leg_24 = plot(framestyle=:none, grid=false, showaxis=false,
                      size=(520,130), dpi=300, background_color=:transparent)
        scatter!(leg_24,[],[]; markershape=:square, markersize=12,
                markercolor=:red, alpha=0.8, markerstrokewidth=0,
                label="Experimental (mean\u00b1SD)")
        plot!(leg_24,[],[]; color=:black, linewidth=3, marker=:circle,
              markersize=10, markercolor=:black, markerstrokewidth=0,
              label="Predicted")
        plot!(leg_24; legend=:top, legendcolumns=2, legendfont=font(13, BF))
        save_both(leg_24, "Fireproof_Validation/dynamics_24h_legend")
    end
end

println("\n All plots complete!")